<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/phonchi/CryoParticleSegment/blob/main/notebook/00-A_Mask_Generator_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

## ⭐ Setup
You must run all codes under this category.

### ✅ Packages Handling

In [1]:
# @title  { display-mode: "form" }
# @markdown Built-in packages.

import os
import shutil
import glob
import tarfile
import time
import subprocess
import socket

In [2]:
# @title  { display-mode: "form" }
# @markdown Useful packages.

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
try:
  from google.colab import files
  from google.colab import userdata
except:
  pass

In [3]:
# @title  { display-mode: "form" }
# @markdown Other required packages.

%pip install wget -qq
import wget
%pip install starfile -qq
import starfile

  Preparing metadata (setup.py) ... done


In [4]:
# @title  { display-mode: "form" }
# @markdown Cryo-em packages.

%pip install mrcfile -qq
import mrcfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 3.7 MB/s eta 0:00:00


### ✅ Function

In [5]:
# @title  { display-mode: "form" }
# @markdown Operating system

def get_basename(path):
  return os.path.basename(path)

def get_basename_with_uid_removed(path):
  return os.path.basename(path).split(sep='_', maxsplit=1)[-1]

def change_root(path, root):
  basename = os.path.basename(path)
  return os.path.join(root, basename)

def change_root_func(root):
  def func(path):
    basename = os.path.basename(path)
    return os.path.join(root, basename)
  return func

def change_root_with_particle_id_remain(root):
  def func(path):
    particle_id = path.split('@')[0]
    basename = os.path.basename(path)
    return os.path.join(particle_id+'@'+root, basename)
  return func

In [6]:
# @title  { display-mode: "form" }
# @markdown Data preparation

def download_extract_empiar(save_to, empiar_id, url):
  """
  Download the EMPIAR-XXXXX.tar.g to `save_to`,
  where the XXXXX is the empiar_id.
  Extract the EMPIAR-XXXXX.tar.gz after downloading.
  """

  empiar_dir = os.path.join(save_to, f"EMPIAR-{empiar_id}")

  if not os.path.isdir(empiar_dir):
    downloaded_path = os.path.join(save_to, f"{empiar_id}")
    try:
      os.mkdir(downloaded_path)
      if not os.path.exists(f"{TMP_DIR}/{empiar_id}.tar.gz"):
        print(f"Downloading {empiar_id}.tar.gz.")
        wget.download(f"{url}/{empiar_id}.tar.gz", f"{TMP_DIR}/{empiar_id}.tar.gz")
      # extract EMPIAR-XXXXX.tar.gz, same utility as
      with tarfile.open(f"{TMP_DIR}/{empiar_id}.tar.gz") as tar:
        print(f"Extracting {empiar_id}.tar.gz.")
        tar.extractall(path=save_to, filter='data')
    except FileExistsError:
      print(f"{empiar_id} exist but not rename to EMPIAR-{empiar_id}.")
    finally:
      os.rename(downloaded_path, empiar_dir)

  return empiar_dir

def get_particles_by_starfile(star_file, micrograph_dir, particle_dir=None):
    particle_selected = starfile.read(star_file)
    try:
      optics = particle_selected['optics']
      particles = particle_selected['particles']
    except:
      particles = particle_selected
    rlnMicrographName = particles.pop('rlnMicrographName')
    rlnImageName = particles.pop('rlnImageName')
    columns = particles.columns
    try:
        particles["rlnMicrographName"] = rlnMicrographName.apply(os.path.basename)
        particles["MicrographName"] = particles['MicrographName'].apply(lambda x: x.split(sep='_', maxsplit=1)[-1])
    except:
        pass
    try:
        particles["ImageName"] = rlnImageName.apply(os.path.basename)
    except:
        pass
    micrograph_names = sorted(particles["MicrographName"].unique())
    micrograph_path = []
    particle_paths = []
    coordinate = []
    for micrograph_name in micrograph_names:
        particle = particles[particles["MicrographName"]==micrograph_name]
        micrograph_path.append(os.path.join(micrograph_dir, particle["MicrographName"].iloc[0]))
        particle_paths.append(os.path.join(particle_dir, particle["ImageName"].iloc[0]))
        coordinate.append(particle[columns])
        coordinate[-1].reset_index()
    return (
        optics,
        pd.DataFrame({
            "Micrograph": micrograph_path,
            "Particle": particle_paths,
            "Coordinate": coordinate,
        },
        index=micrograph_names)
    )

### ✅ Directory Settings

In [7]:
# @title  { display-mode: "form" }
# @markdown Directory parameters.

DATA_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data" # @param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_label_0A" # @param {type:"string"}
TMP_DIR = "/content/tmp" # @param {type:"string"}
SOFTWARE_DIR = "/content/software" # @param {type:"string"}
WORK_DIR = os.getcwd()

In [8]:
# @title  { display-mode: "form" }
# @markdown Run this block if using folder📁 in Google Drive as **`RESULT_DIR`**.

try:
  from google.colab import drive
  drive.mount('/content/drive')
except:
  pass

Mounted at /content/drive


In [9]:
# @title  { display-mode: "form" }
# @markdown Run this block for remove the **`sample_data`** folder📁 in content

if os.path.isdir("/content/sample_data"):
  !rm -r /content/sample_data
# from shutil import rmtree
#
# rmtree(f"/content/sample_data")

In [10]:
# @title  { display-mode: "form" }
# @markdown Run this block for checking the existence of the directories

if not os.path.isdir(DATA_DIR):
  os.mkdir(DATA_DIR)

if not os.path.isdir(RESULT_DIR):
  os.mkdir(RESULT_DIR)

if not os.path.isdir(TMP_DIR):
  os.mkdir(TMP_DIR)

if not os.path.isdir(SOFTWARE_DIR):
  os.mkdir(SOFTWARE_DIR)

### ✅ EMPIAR Data Setting

> #### 🗒 Info
> Please refer to the [table here](https://github.com/BioinfoMachineLearning/cryoppp?tab=readme-ov-file#cryoppp-statistics) for the available datasets in CryoPPP and their statistics.  
> For additional details about micrograph features, see the [table in the paper](https://static-content.springer.com/esm/art%3A10.1038%2Fs41597-023-02280-2/MediaObjects/41597_2023_2280_MOESM4_ESM.xlsx).


In [11]:
# @title  { vertical-output: true, display-mode: "form" }
EMPIAR_ID = 10017 # @param {type:"integer"}

In [12]:
EMPIAR_DIR = os.path.join(DATA_DIR, f"EMPIAR-{EMPIAR_ID}")
EMPIAR_ID, DATA_DIR, RESULT_DIR

(10017,
 '/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data',
 '/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_label_0A')

In [13]:
# @title  { vertical-output: true, display-mode: "form" }
# @markdown Data Downloading.

url = "https://calla.rnet.missouri.edu/cryoppp" # @param ["https://calla.rnet.missouri.edu/cryoppp"]
use_diretory = f"/{RESULT_DIR}/dataset/{EMPIAR_ID}/" # @param

if os.path.exists(use_diretory):
  start_time = time.time()
  !mkdir {DATA_DIR}/{EMPIAR_ID}
  !cp -r {use_diretory}/* {DATA_DIR}/{EMPIAR_ID}
  os.rename(os.path.join(DATA_DIR, f"{EMPIAR_ID}"),
            os.path.join(DATA_DIR, f"EMPIAR-{EMPIAR_ID}"))
  EMPIAR_DIR = os.path.join(DATA_DIR, f"EMPIAR-{EMPIAR_ID}")
  end_time = time.time()
  print("Time used:", end_time-start_time)
else:
  start_time = time.time()
  EMPIAR_DIR = download_extract_empiar(DATA_DIR, EMPIAR_ID, url=url)
  end_time = time.time()
  print("Time used:", end_time-start_time)

Time used: 586.4864029884338


In [14]:
!mkdir {RESULT_DIR}/dataset/
!mkdir {RESULT_DIR}/dataset/{EMPIAR_ID}

mkdir: cannot create directory ‘/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_label_0A/dataset/’: File exists
mkdir: cannot create directory ‘/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_label_0A/dataset/10017’: File exists


In [15]:
# @title  { vertical-output: true, display-mode: "form" }
# @markdown Assigning processed data directory.
PROCESSED_DIR = f"{EMPIAR_DIR}/processed"

if not os.path.isdir(PROCESSED_DIR):
  os.mkdir(PROCESSED_DIR)

print(f"PROCESSED_DIR: {PROCESSED_DIR}")

PROCESSED_DIR: /content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data/EMPIAR-10017/processed


## ⏭ Software Installation
You can choose the software you want according to your needs.

#### 🟪 Topaz Installation

In [16]:
# @title  { display-mode: "form" }

INSTALL = True # @param {type:"boolean"}

if INSTALL:
  %cd {SOFTWARE_DIR}
  !git clone https://github.com/tbepler/topaz.git
  %cd topaz
  !pip install -e .
  %cd ..

/content/software
Cloning into 'topaz'...
remote: Enumerating objects: 3548, done.
remote: Counting objects: 100% (666/666), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 3548 (delta 627), reused 570 (delta 570), pack-reused 2882 (from 2)
Receiving objects: 100% (3548/3548), 253.12 MiB | 17.04 MiB/s, done.
Resolving deltas: 100% (2320/2320), done.
/content/software/topaz
Obtaining file:///content/software/topaz
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for topaz-em (pyproject.toml) ... done
  Created wheel for topaz-em: filename=topaz_em-0.3.17-0.editable-py3-none-any.whl size=23640 sha256=95be80f8d15c287d676dbae4dd1ba24a67e0331d8fa962ef9e78553949ac383f
  Stored in directory: /tmp/pip-ephem-wheel-cache-pr1fd5dv/wheels/6a/5e/15/fac72173c4f55fab5caeb83eb85209b35727ffa7a5df74f521
Suc

#### 🟪 CryoSPARC Installation

In [17]:
# @title  { display-mode: "form" }

INSTALL = True # @param {type:"boolean"}

> #### 🗒 Info
> To proceed, please obtain a LICENSE ID from [this page](https://cryosparc.com/download).


In [18]:
# @title  { display-mode: "form" }
# @markdown Download and extract CryoSPARC.

if INSTALL:
  try:
    LICENSE_ID = '9657b740-93cb-11f0-a34d-cb5f4bdb076d' # @param {type:"string"}
  except:
    pass

  os.environ['LICENSE_ID'] = LICENSE_ID

  %cd {SOFTWARE_DIR}
  %mkdir cryosparc
  %cd cryosparc
  if not os.path.isdir("cryosparc_master"):
    if not os.path.exists("cryosparc_master.tar.gz"):
      !curl -L https://get.cryosparc.com/download/master-latest/$LICENSE_ID -o cryosparc_master.tar.gz
    !tar -xf cryosparc_master.tar.gz cryosparc_master
    !rm /content/software/cryosparc/cryosparc_master.tar.gz
  if not os.path.isdir("cryosparc_worker"):
    if not os.path.exists("cryosparc_worker.tar.gz"):
      !curl -L https://get.cryosparc.com/download/worker-latest/$LICENSE_ID -o cryosparc_worker.tar.gz
    !tar -xf cryosparc_worker.tar.gz cryosparc_worker
    !rm /content/software/cryosparc/cryosparc_worker.tar.gz

/content/software
/content/software/cryosparc
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:--  0:00:03 --:--:--     0
100  630M  100  630M    0     0  12.0M      0  0:00:52  0:00:52 --:--:-- 13.3M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:--  0:00:01 --:--:--     0
100 3993M  100 3993M    0     0  13.0M      0  0:05:05  0:05:05 --:--:-- 15.5M


> #### ⚠ Notice
> During installation, you will be prompted with several questions. Simply answer "Yes" to proceed.


In [ ]:
# @title { display-mode: "form" }
# @markdown ## Install CryoSPARC

INSTALL = True  # @param {type:"boolean"}
EMAIL = "yki.016@g-mail.nsysu.edu.tw"  # @param {type:"string"}
PASSWORD = "password"  # @param {type:"string"}
USERNAME = "ANIMATH"  # @param {type:"string"}
FIRST_NAME = "Sin-Rong"  # @param {type:"string"}
LAST_NAME = "Tsai"  # @param {type:"string"}
PORT = 61000  # @param {type:"integer"}

import os

if INSTALL:
  %cd cryosparc_master
  !apt-get install iputils-ping -y
  %mkdir -p /content/software/cryosparc/cryosparc_cache

  os.environ['LICENSE_ID'] = LICENSE_ID
  os.environ['USER'] = USERNAME
  %env CRYOSPARC_FORCE_USER=true

  !./install.sh --standalone \
      --license $LICENSE_ID \
      --worker_path /content/software/cryosparc/cryosparc_worker \
      --ssdpath /content/software/cryosparc/cryosparc_cache \
      --initial_email "{EMAIL}" \
      --initial_password "{PASSWORD}" \
      --initial_username "{USERNAME}" \
      --initial_firstname "{FIRST_NAME}" \
      --initial_lastname "{LAST_NAME}" \
      --port {PORT} \
      --yes
  %cd {WORK_DIR}

/content/software/cryosparc/cryosparc_master
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  iputils-ping
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 43.0 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 iputils-ping amd64 3:20211215-1ubuntu0.1 [43.0 kB]
Fetched 43.0 kB in 1s (50.0 kB/s)
Selecting previously unselected package iputils-ping.
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack .../iputils-ping_3%3a20211215-1ubuntu0.1_amd64.deb ...
Unpacking iputils-ping (3:20211215-1ubuntu0.1) ...
Setting up iputils-ping (3:20211215-1ubuntu0.1) ...
Processing triggers for man-db (2.10.2-1) ...
env: CRYOSPARC_FORCE_USER=true

************ CRYOSPARC SYSTEM: STANDALONE INSTALLER **************

You are the root user. Are you s

In [ ]:
# @title  { display-mode: "form" }
# @markdown Install pyngrok.

if INSTALL:
  %pip install pyngrok -qq

#### 🟪 Pyem Installation

In [ ]:
# @title  { display-mode: "form" }

INSTALL = True # @param {type:"boolean"}

if INSTALL:
  %cd {SOFTWARE_DIR}
  !pip install pyFFTW
  !pip install healpy
  !pip install pathos
  !git clone https://github.com/asarnow/pyem.git
  %cd pyem
  !pip install --no-dependencies -e .
  %cd {WORK_DIR}

#### 🟪 Relion Installation

In [ ]:
# @title  { display-mode: "form" }
INSTALL = True # @param {type:"boolean"}

if INSTALL:
  %cd {SOFTWARE_DIR}
  !sudo apt install libfftw3-dev
  !git clone https://github.com/3dem/relion.git
  %cd relion
  !git checkout master # or ver4.0
  %mkdir build
  %cd build
  !cmake ..
  !make -j8
  !make install
  %cd {WORK_DIR}

## ⏭ Data Preparation

### 🟪 Micrograph preprocessing (by topaz).

In [ ]:
# @title  { vertical-output: true, display-mode: "form" }
# @markdown Preprocess the micrograph using topaz.

topaz_preprocessing = True # @param {type:"boolean"}
save_preprocessing = True # @param {type:"boolean"}

#if not os.path.isdir(os.path.join(PROCESSED_DIR, "micrographs")):
if topaz_preprocessing:
    !topaz preprocess -v -o {PROCESSED_DIR}/micrographs/ {EMPIAR_DIR}/micrographs/*.mrc
    if save_preprocessing:
        !mkdir {RESULT_DIR}/dataset/{EMPIAR_ID}/processed_micrographs
        !rsync -av {PROCESSED_DIR}/micrographs/* {RESULT_DIR}/dataset/{EMPIAR_ID}/processed_micrographs/
else:
    start_time = time.time()
    !mkdir {PROCESSED_DIR}/micrographs
    !rsync -av {RESULT_DIR}/dataset/{EMPIAR_ID}/processed_micrographs/* {PROCESSED_DIR}/micrographs/
    end_time = time.time()
    print("Time used:", end_time-start_time)
    # !unzip -xf {RESULT_DIR}/processed_micrographs.zip {PROCESSED_DIR}/micrographs

### 🟪 Handling `selected.star` and `particles.txt` (by topaz).

In [ ]:
# @title  { display-mode: "form" }
# @markdown Try to move those files from **`RESULT DIR`** 📁 to **`PROCESSED DIR`** 📁.

try:
  shutil.copy(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/selected.star", PROCESSED_DIR)
  selected_star_exist = True
except:
  print(f"No selected.star found in {RESULT_DIR}/dataset/{EMPIAR_ID}/")
  selected_star_exist = False

In [ ]:
# @markdown Generate selected particle starfile that is suitable for topaz (**`selected.star`**) if it is not in **`RESULT DIR`** 📁. { display-mode: "form" }

# @markdown Use **`selected.star`** from **`RESULT DIR`** 📁.
use_result = False # @param {type:"boolean"}
if use_result:
  try:
    shutil.copy(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/selected.star", f"{PROCESSED_DIR}")
  except:
    pass

# @markdown Force to generate a new **`selected.star`**.
FORCE = True # @param {type:"boolean"}
if FORCE or not os.path.exists(f"{PROCESSED_DIR}/selected.star"):
  remove_root = True # @param {type:"boolean"}
  SAVE = True # @param {type:"boolean"}

  df = starfile.read(f"{EMPIAR_DIR}/ground_truth/empiar-{EMPIAR_ID}_particles_selected.star")
  df_optics, df_particles = df['optics'].copy(), df['particles'].copy()
  if remove_root:
    df_particles['rlnImageName'] = df_particles['rlnImageName'].apply(get_basename)
    df_particles['rlnMicrographName'] = df_particles['rlnMicrographName'].apply(get_basename_with_uid_removed)
  else:
    df_particles['rlnImageName'] = df_particles['rlnImageName'].apply(change_root_with_particle_id_remain(f"{EMPIAR_DIR}/particles_stack"))
    df_particles['rlnMicrographName'] = df_particles['rlnMicrographName'].apply(change_root_func(f"{PROCESSED_DIR}/micrographs"))

  starfile.write(df_particles, f"{PROCESSED_DIR}/selected.star")
  if SAVE:
    shutil.copy(f"{PROCESSED_DIR}/selected.star", f"{RESULT_DIR}/dataset/{EMPIAR_ID}")

In [ ]:
# Remove root!
df_particles

In [ ]:
# @markdown ⛔ Problem (Solved by running this cell.)

# @markdown If using ` --invert-y`, `--image-ext` need to be set to `mrc`,
# @markdown and `coords = coords.clone()` in
# @markdown "/content/software/topaz/topaz/utils/conversions.py"
# @markdown need to be change to `coords = coords.copy()`
%%writefile /content/software/topaz/topaz/utils/conversions.py

from __future__ import division, print_function

import glob
import json
import os
import sys
from locale import strcoll
from typing import List

import numpy as np
import pandas as pd
import topaz.utils.star as star
from topaz.utils.data.loader import load_image

def mirror_y_axis(coords, n):
    coords = coords.copy()
    coords['y_coord'] = n-1-coords['y_coord']
    return coords

def boxes_to_coordinates(boxes, shape=None, invert_y=False, image_name=None):
    if len(boxes) < 1: # boxes are empty, return empty coords table
        columns = ['x_coord', 'y_coord']
        if image_name is not None:
            columns.append('image_name')
        coords = pd.DataFrame(columns=columns)
        return coords

    ## first 2 columns are x and y coordinates of lower left box corners
    ## next 2 columns are width and height

    ## requires knowing image size to invert y-axis (shape parameter)
    ## to conform with origin in upper-left rather than lower-left
    ## apparently, EMAN2 only inverts the y-axis for .tiff images
    ## so box files only need to be inverted when working with .tiff
    x_lo = boxes[:,0]
    y_lo = boxes[:,1]
    width = boxes[:,2]
    height = boxes[:,3]
    x_coord = x_lo + width//2
    y_coord = y_lo + height//2

    if invert_y:
        y_coord = (shape[0]-1-y_lo) - height//2

    coords = np.stack([x_coord, y_coord], axis=1)
    if image_name is not None: # in this case, return as table with image_name column
        coords = pd.DataFrame(coords, columns=['x_coord', 'y_coord'])
        coords.insert(0, 'image_name', [image_name]*len(coords))

    return coords


def file_boxes_to_coordinates(input_paths:List[str], image_dir:str, image_ext:str, invert_y:bool, output_path:str=None):
    tables = []

    for path in input_paths:
        if os.path.getsize(path) == 0:
            continue

        shape = None
        image_name = os.path.splitext(os.path.basename(path))[0]
        if invert_y:
            impath = os.path.join(image_dir, image_name) + '.' + image_ext
            # use glob incase image_ext is '*'
            impath = glob.glob(impath)[0]
            im = load_image(impath)
            shape = (im.height,im.width)

        box = pd.read_csv(path, sep='\t', header=None).values

        coords = boxes_to_coordinates(box, shape=shape, invert_y=invert_y, image_name=image_name)

        tables.append(coords)

    table = pd.concat(tables, axis=0)

    output = sys.stdout if output_path is None else output_path
    table.to_csv(output, sep='\t', index=False)


def coordinates_to_boxes(coords, box_width, box_height, shape=None, invert_y=False, tag='manual'):
    entries = []
    x_coords = coords[:,0]
    y_coords = coords[:,1]
    if invert_y:
        y_coords = shape[0]-1-coords[:,1]
    box_width = np.array([box_width]*len(x_coords), dtype=np.int32)
    box_height = np.array([box_height]*len(x_coords), dtype=np.int32)

    # x and y are centers, make lower left corner
    x_coords = x_coords - box_width//2
    y_coords = y_coords - box_height//2

    boxes = np.stack([x_coords, y_coords, box_width, box_height], 1)
    return boxes


def file_coordinates_to_boxes(input_paths:List[str], destdir:str, boxsize:int, invert_y:bool, image_dir:str, image_ext:str):
    dfs = []
    for path in input_paths:
        coords = pd.read_csv(path, sep='\t')
        dfs.append(coords)
    coords = pd.concat(dfs, axis=0)

    coords = coords.drop_duplicates()

    if not os.path.exists(destdir):
        os.makedirs(destdir)

    for image_name,group in coords.groupby('image_name'):
        path = destdir + '/' + image_name + '.box'

        shape = None
        if invert_y:
            impath = os.path.join(image_dir, image_name) + '.' + image_ext
            # use glob incase image_ext is '*'
            impath = glob.glob(impath)[0]
            im = load_image(impath)
            shape = (im.height,im.width)

        xy = group[['x_coord', 'y_coord']].values.astype(np.int32)

        boxes = coordinates_to_boxes(xy, boxsize, boxsize, shape=shape, invert_y=invert_y)
        boxes = pd.DataFrame(boxes)

        boxes.to_csv(path, sep='\t', header=False, index=False)


def coordinates_to_eman2_json(coords, shape=None, invert_y=False, tag='manual'):
    entries = []
    x_coords = coords[:,0]
    y_coords = coords[:,1]
    if invert_y:
        y_coords = shape[0]-1-coords[:,1]
    for i in range(len(x_coords)):
        entries.append([int(x_coords[i]), int(y_coords[i]), tag])
    return entries


def file_coordinates_to_eman2_json(input_paths:List[str], destdir:str, invert_y:bool, image_dir:str, image_ext:str):
    dfs = []
    for path in input_paths:
        coords = pd.read_csv(path, sep='\t')
        dfs.append(coords)
    coords = pd.concat(dfs, axis=0)

    coords = coords.drop_duplicates()
    print(len(coords))

    if not os.path.exists(destdir):
        os.makedirs(destdir)

    for image_name,group in coords.groupby('image_name'):
        path = destdir + '/' + image_name + '_info.json'

        shape = None
        if invert_y:
            impath = os.path.join(image_dir, image_name) + '.' + image_ext
            # use glob incase image_ext is '*'
            impath = glob.glob(impath)[0]
            im = load_image(impath)
            shape = (im.height,im.width)

        xy = group[['x_coord','y_coord']].values.astype(int)
        boxes = coordinates_to_eman2_json(xy, shape=shape, invert_y=invert_y)

        with open(path, 'w') as f:
            json.dump({'boxes': boxes}, f, indent=0)


def coordinates_to_star(table, image_ext=''):
    # fix column names to be star format
    d = {'score': star.SCORE_COLUMN_NAME,
            'image_name': 'MicrographName',
            'x_coord': star.X_COLUMN_NAME,
            'y_coord': star.Y_COLUMN_NAME,
            'voltage': star.VOLTAGE,
            'detector_pixel_size': star.DETECTOR_PIXEL_SIZE,
            'magnification': star.MAGNIFICATION,
            'amplitude_contrast': star.AMPLITUDE_CONTRAST,
            }
    table = table.copy()
    for k,v in d.items():
        if k in table.columns:
            table[v] = table[k]
            table = table.drop(k, axis=1)
    # append image extension
    table['MicrographName'] = table['MicrographName'].apply(lambda x: x + image_ext)

    return table


def star_to_coordinates(input_file, output_file=None):
    def strip_ext(name):
        clean_name,ext = os.path.splitext(name)
        return clean_name

    with open(input_file, 'r') as f:
        table = star.parse(f)

    if 'ParticleScore' in table.columns:
        ## columns of interest are 'MicrographName', 'CoordinateX', 'CoordinateY', and 'ParticleScore'
        table = table[['MicrographName', 'CoordinateX', 'CoordinateY', 'ParticleScore']]
        table.columns = ['image_name', 'x_coord', 'y_coord', 'score']
    else:
        ## columns of interest are 'MicrographName', 'CoordinateX', and 'CoordinateY'
        table = table[['MicrographName', 'CoordinateX', 'CoordinateY']]
        table.columns = ['image_name', 'x_coord', 'y_coord']
    ## convert the coordinates to integers
    table['x_coord'] = table['x_coord'].astype(float).astype(int)
    table['y_coord'] = table['y_coord'].astype(float).astype(int)
    ## strip file extensions off the image names if present
    table['image_name'] = table['image_name'].apply(strip_ext)

    out = output_file if output_file is not None else sys.stdout
    table.to_csv(out, sep='\t', index=False)


In [ ]:
# @markdown ⛔ Problem (Solved by running this cell.)

# @markdown If using ` --invert-y`, `--image-ext` need to be set to `mrc`,
# @markdown "/content/software/topaz/topaz/commands/convert.py"
# @markdown need to be change to `im.height -> im[0].height`
%%writefile /content/software/topaz/topaz/commands/convert.py

from __future__ import print_function,division

import sys
import os
import glob
import pandas as pd
import numpy as np
import argparse

import topaz.utils.star as star
import topaz.utils.files as file_utils
from topaz.utils.conversions import mirror_y_axis
from topaz.utils.data.loader import load_image


name = 'convert'
help = 'convert particle coordinate files between various formats automatically. also allows filtering particles by score threshold and UP- and DOWN-scaling coordinates.'


def add_arguments(parser=None):
    # parser = argparse.ArgumentParser('Script to ' + help)
    if parser is None:
        parser = argparse.ArgumentParser(help)

    parser.add_argument('files', nargs='+', help='path to input particle file(s). when multiple input files are given, they are concatentated into a single output file.')
    parser.add_argument('-o', '--output', help='path to output particle file (default: stdout)')

    parser.add_argument('--from', dest='_from', choices=['auto', 'coord', 'csv', 'star', 'box'], default='auto'
                       , help='file format of the INPUT file (default: detect format automatically based on file extension)')
    parser.add_argument('--to', choices=['auto', 'coord', 'csv', 'star', 'json', 'box'], default='auto'
                       , help='file format of the OUTPUT file. NOTE: when converting to JSON or BOX formats, OUTPUT must specify the destination directory. (default: detect format automatically based on file extension)')

    parser.add_argument('--suffix', default='', help='suffix to append to file names when writing to directory (default: none)')

    # arguments for thresholding/scaling coordinates
    parser.add_argument('-t', '--threshold', type=float, default=-np.inf, help='threshold the particles by score (optional)')
    parser.add_argument('-s', '--down-scale', type=float, default=1, help='DOWN-scale coordinates by this factor. new coordinates will be coord_new = (x/s)*coord_cur. (default: 1)')
    parser.add_argument('-x', '--up-scale', type=float, default=1, help='UP-scale coordinates by this factor. new coordinates will be coord_new = (x/s)*coord_cur. (default: 1)')

    # metadata arguments that can be added to particle files
    parser.add_argument('--voltage', type=float, default=-1, help='voltage metadata (optional)')
    parser.add_argument('--detector-pixel-size', type=float, default=-1, help='detector pixel size metadata (optional)')
    parser.add_argument('--magnification', type=float, default=-1, help='magnification metadata (optional)')
    parser.add_argument('--amplitude-contrast', type=float, default=-1, help='amplitude contrast metadata (optional)')

    # arguments for file format specific parameters
    parser.add_argument('--invert-y', action='store_true', help='invert (mirror) the y-axis particle coordinates. requires also specifying --imagedir.')
    parser.add_argument('--imagedir', help='directory of images. only required to invert the y-axis - sometimes necessary for particles picked on .tiff images')
    parser.add_argument('--image-ext', default='.mrc', help='image file extension. required for converting to STAR and BOX formats and to find images when --invert-y is set. (default=.mrc)')
    parser.add_argument('--boxsize', default=0, type=int, help='size of particle boxes. required for converting to BOX format.')

    # verbose output?
    parser.add_argument('-v', '--verbose', type=int, default=0, help='verbosity of information printed (default: 0)')

    return parser

def main(args):

    verbose = args.verbose

    form = args._from
    from_forms = [form for _ in range(len(args.files))]

    # detect the input file formats
    if form == 'auto':
        try:
            from_forms = [file_utils.detect_format(path) for path in args.files]
        except file_utils.UnknownFormatError as e:
            print('Error: unrecognized input coordinates file extension ('+e.ext+')', file=sys.stderr)
            sys.exit(1)
    formats_detected = list(set(from_forms))
    if verbose > 0:
        print('# INPUT formats detected: '+str(formats_detected), file=sys.stderr)

    # determine the output file format
    output_path = args.output
    output = None
    to_form = args.to
    if output_path is None:
        output = sys.stdout
        # if output is to stdout and form is not set
        # then raise an error
        if to_form == 'auto':
            if len(formats_detected) == 1:
                # write the same output format as input format
                to_form = from_forms[0]
            else:
                print('Error: writing file to stdout and multiple input formats present with no output format (--to) set! Please tell me what format to write!')
                sys.exit(1)
        if to_form == 'box' or to_form == 'json':
            print('Error: writing BOX or JSON output files requires a destination directory. Please set the --output parameter!')
            sys.exit(1)

    image_ext = args.image_ext
    boxsize = args.boxsize
    if to_form == 'auto':
        # first check for directory
        if output_path[-1] == '/':
            # image-ext must be set for these file formats
            if image_ext is None:
                print('Error: writing BOX or JSON output files requires setting the image file extension!')
                sys.exit(1)
            # format is either json or box, check for boxsize to decide
            if boxsize > 0:
                # write boxes!
                if verbose > 0:
                    print('# Detected output format is BOX, because OUTPUT is a directory and boxsize > 0.', file=sys.stderr)
                to_form = 'box'
            else:
                if verbose > 0:
                    print('# Detected output format is JSON, because OUTPUT is a directory and no boxsize set.', file=sys.stderr)
                to_form = 'json'
        else:
            try:
                to_form = file_utils.detect_format(output_path)
            except file_utils.UnknownFormatError as e:
                print('Error: unrecognized output coordinates file extension ('+e.ext+')', file=sys.stderr)
                sys.exit(1)
    if verbose > 0:
        print('# OUTPUT format: ' + to_form)

    suffix = args.suffix

    t = args.threshold
    down_scale = args.down_scale
    up_scale = args.up_scale
    scale = up_scale/down_scale

    # special case when inputs and outputs are all star files
    if len(formats_detected) == 1 and formats_detected[0] == 'star' and to_form == 'star':
        dfs = []
        for path in args.files:
            with open(path, 'r') as f:
                table = star.parse(f)
            dfs.append(table)
        table = pd.concat(dfs, axis=0)
        # convert  score column to float and apply threshold
        if star.SCORE_COLUMN_NAME in table.columns:
            table = table.loc[table[star.SCORE_COLUMN_NAME] >= t]
        # scale coordinates
        if scale != 1:
            x_coord = table[star.X_COLUMN_NAME].values
            x_coord = np.round(scale*x_coord).astype(int)
            table[star.X_COLUMN_NAME] = x_coord
            y_coord = table[star.Y_COLUMN_NAME].values
            y_coord = np.round(scale*y_coord).astype(int)
            table[star.Y_COLUMN_NAME] = y_coord
        # add metadata if specified
        if args.voltage > 0:
            table[star.VOLTAGE] = args.voltage
        if args.detector_pixel_size > 0:
            table[star.DETECTOR_PIXEL_SIZE] = args.detector_pixel_size
        if args.magnification > 0:
            table[star.MAGNIFICATION] = args.magnification
        if args.amplitude_contrast > 0:
            table[star.AMPLITUDE_CONTRAST] = args.amplitude_contrast
        # write output file
        if output is None:
            with open(output_path, 'w') as f:
                star.write(table, f)
        else:
            star.write(table, output)


    else: # general case

        # read the input files
        dfs = []
        for i in range(len(args.files)):
            path = args.files[i]
            coords = file_utils.read_coordinates(path, format=from_forms[i])
            dfs.append(coords)
        coords = pd.concat(dfs, axis=0)

        # threshold particles by score (if there is a score)
        if 'score' in coords.columns:
            coords = coords.loc[coords['score'] >= t]

        # scale coordinates
        if scale != 1:
            x_coord = coords['x_coord'].values
            x_coord = np.round(scale*x_coord).astype(int)
            coords['x_coord'] = x_coord
            y_coord = coords['y_coord'].values
            y_coord = np.round(scale*y_coord).astype(int)
            coords['y_coord'] = y_coord

        # add metadata if specified
        if args.voltage > 0:
            coords['voltage'] = args.voltage
        if args.detector_pixel_size > 0:
            coords['detector_pixel_size'] = args.detector_pixel_size
        if args.magnification > 0:
            coords['magnification'] = args.magnification
        if args.amplitude_contrast > 0:
            coords['amplitude_contrast'] = args.amplitude_contrast

        # invert y-axis coordinates if specified
        invert_y = args.invert_y
        if invert_y:
            if args.imagedir is None:
                print('Error: --imagedir must specify the directory of images in order to mirror the y-axis coordinates', file=sys.stderr)
                sys.exit(1)
            dfs = []
            for image_name,group in coords.groupby('image_name'):
                impath = os.path.join(args.imagedir, image_name) + '.' + args.image_ext
                # use glob incase image_ext is '*'
                impath = glob.glob(impath)[0]
                im = load_image(impath)
                height = im[0].height

                group = mirror_y_axis(group, height)
                dfs.append(group)
            coords = pd.concat(dfs, axis=0)

        # output file format is decided and coordinates are processed, now write files
        if output is None and to_form != 'box' and to_form != 'json':
            output = open(output_path, 'w')
        if to_form == 'box' or to_form == 'json':
            output = output_path

        file_utils.write_coordinates(output, coords, format=to_form, boxsize=boxsize, image_ext=image_ext, suffix=suffix)


if __name__ == '__main__':
    parser = add_arguments()
    args = parser.parse_args()
    main(args)


In [ ]:
# @title  { display-mode: "form" }
# @markdown Generate particle location txtfile (**`particles.txt`**).

# error: https://github.com/tbepler/topaz/issues/60
# If using ` --invert-y`, set --image-ext with `mrc`,
# and change `coords = coords.clone()` in
# "/usr/local/lib/python3.10/dist-packages/topaz/utils/conversions.py"
# to `coords = coords.copy()`

# @markdown Whether to try using **`particles.txt`** from **`RESULT DIR`**📁.
use_result = False # @param {type:"boolean"}
file_exist = os.path.exists(f"{PROCESSED_DIR}/particles.txt")
if use_result:
  try:
    shutil.copy(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/particles.txt", PROCESSED_DIR)
    file_exist = True
  except:
    print(f"No particles.txt found in {RESULT_DIR}/dataset/{EMPIAR_ID}/")
    file_exist = False

# @markdown Force to generate a new **`particles.txt`**.
FORCE = True # @param {type:"boolean"}
# if FORCE:
#   !rm {RESULT_DIR}/particles.txt
if FORCE or not file_exist:
  # ground_truth_star_path = f"{EMPIAR_DIR}/ground_truth/empiar-{EMPIAR_ID}_particles_selected.star"
  !topaz convert --invert-y --image-ext mrc\
          --imagedir {PROCESSED_DIR}/micrographs/ \
          --from star -o {PROCESSED_DIR}/particles.txt \
           {PROCESSED_DIR}/selected.star
  # {ground_truth_star_path}

  # @markdown Whether to save result to **`RESULT DIR`**📁.
  SAVE = True # @param {type:"boolean"}
  if SAVE:
    shutil.copy(f"{PROCESSED_DIR}/particles.txt", f"{RESULT_DIR}/dataset/{EMPIAR_ID}/")

## ⏭ Particles Projection

In [ ]:
# @title  { run: "auto", vertical-output: true, display-mode: "form" }

# @markdown If this option is set to true, the projection of particle (.mrcs) will be created.

# @markdown Files required within the process will be automatically detected and generated.
# @markdown Note that you need to install the corresponding software to guarantee the performance.

project_particles = True # @param {type:"boolean"}
if project_particles:
  if os.path.exists(f"{PROCESSED_DIR}/proj_{EMPIAR_ID}.mrcs"):
    print(f"File exists: {PROCESSED_DIR}/proj_{EMPIAR_ID}.mrcs")
    project_particles = input("Do you sure you want to project particles? y/n\n").lower()=="y"
  elif os.path.exists(f"{PROCESSED_DIR}/proj_{EMPIAR_ID}.star"):
    print(f"File exists: {PROCESSED_DIR}/proj_{EMPIAR_ID}.star")
    project_particles = input("Do you sure you want to project particles? y/n\n").lower()=="y"
  if project_particles:
    print("`project_particles` is set to `True`.")
  else:
    print("`project_particles` is still `False`.")


#### Split the data set

In [ ]:
# @markdown You can also choose to use a subset of particles to obtain the 3D mask

user = True # @param {type:"boolean"}

from sklearn.model_selection import train_test_split
def save_mrc_to_npy(filenames, mrc_dir, save_dir, permissive=False):
  for filename in filenames:
    filepath = os.path.join(mrc_dir, filename)
    print("\rReading mrc:", filepath, end="", flush=True)
    with mrcfile.open(filepath, permissive=permissive) as mrc:
      basename, ext = os.path.splitext(filename)
      assert ext == ".mrc"
      print("\rConverting mrc to npy.", end="", flush=True)
      np.save(os.path.join(save_dir, basename), mrc.data)

MRC_DIR = f'{RESULT_DIR}/dataset/{EMPIAR_ID}/processed_micrographs'
SAVE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_label_0A/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
!mkdir $SAVE_DIR

# @markdown ---
# title Train test split
random_state = None # @param {type:"integer"}
test_size = "0.83" # @param {type:"string"}
val_size = "0.071" # @param {type:"string"}
filenames = sorted(os.listdir(MRC_DIR))

test_size = float(test_size)
if test_size and not 0<=test_size<1:
  raise ValueError(f"`test_size` should be between 0.0 and 1.0, got: {test_size}")
val_size = float(val_size)
if val_size and not 0<=val_size<1:
  raise ValueError(f"`val_size` should be between 0.0 and 1.0, got: {val_size}")
train_size = 1 - test_size - val_size
if not 0<train_size<=1:
  raise ValueError(f"`val_size` + `test_size` should be between 0.0 and 1.0, got: {test_size + val_size}")
train_filenames, test_filenames = train_test_split(filenames, test_size=test_size, train_size=train_size, random_state=3)
train_filenames = sorted(train_filenames)
test_filenames = sorted(test_filenames)
val_filenames = []
for filename in filenames:
  if filename not in train_filenames and filename not in test_filenames:
    val_filenames.append(filename)
print("number of files:",
  f" train:\t{len(train_filenames)}",
  f" test:\t{len(test_filenames)}",
  f" val:\t{len(val_filenames)}", sep='\n')



In [ ]:
%%capture --no-display
# @title Generate np file from mrc files.
# @markdown note that directory for np files should not exist


PERMISSIVE = True # @param {type:"boolean"}

try:
    os.mkdir(SAVE_DIR)
except FileExistsError:
    pass

os.mkdir(os.path.join(SAVE_DIR, "train"))
save_mrc_to_npy(train_filenames,
                mrc_dir=MRC_DIR,
                save_dir=os.path.join(SAVE_DIR, "train"),
                permissive=PERMISSIVE)
np.savetxt(os.path.join(SAVE_DIR, "train_filenames.txt"), train_filenames, fmt="%s")

os.mkdir(os.path.join(SAVE_DIR, "test"))
save_mrc_to_npy(test_filenames, mrc_dir=MRC_DIR,
                save_dir=os.path.join(SAVE_DIR, "test"),
                permissive=PERMISSIVE)
np.savetxt(os.path.join(SAVE_DIR, "test_filenames.txt"), test_filenames, fmt="%s")

os.mkdir(os.path.join(SAVE_DIR, "val"))
save_mrc_to_npy(val_filenames, mrc_dir=MRC_DIR,
                save_dir=os.path.join(SAVE_DIR, "val"),
                permissive=PERMISSIVE)
np.savetxt(os.path.join(SAVE_DIR, "val_filenames.txt"), val_filenames, fmt="%s")

In [ ]:
file_path = os.path.join(SAVE_DIR, 'train_filenames.txt')
file_path2 = os.path.join(SAVE_DIR, 'val_filenames.txt')
file_path3 = os.path.join(SAVE_DIR, 'test_filenames.txt')

with open(file_path, 'r') as f:
    train_filenames = [line.strip() for line in f if line.strip()]

with open(file_path2, 'r') as f:
    val_filenames = [line.strip() for line in f if line.strip()]

with open(file_path3, 'r') as f:
    test_filenames = [line.strip() for line in f if line.strip()]

print("number of files:",
  f" train:\t{len(train_filenames)}",
  f" test:\t{len(test_filenames)}",
  f" val:\t{len(val_filenames)}", sep='\n')

### Provide the split filenames

In [ ]:
# @title  { run: "auto", vertical-output: true, display-mode: "form" }

# @markdown You can also choose to use a subset of particles to obtain the 3D mask

user = True # @param {type:"boolean"}

if user:
    SAVE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_label_0A/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
    # --- User parameters ---
    input_star   = f'{EMPIAR_DIR}/ground_truth/empiar-{EMPIAR_ID}_particles_selected.star'     # path to your input .star file
    output_star  = 'filtered_train.star'      # desired output path
    output_star2  = 'filtered_val.star'

    file_path = os.path.join(SAVE_DIR, 'train_filenames.txt')
    file_path2 = os.path.join(SAVE_DIR, 'val_filenames.txt')
    file_path3 = os.path.join(SAVE_DIR, 'test_filenames.txt')

    # Read lines into a list
    with open(file_path, 'r') as f:
        keep = [line.strip() for line in f if line.strip()]

    # --- Read the entire STAR file ---
    with open(input_star, 'r') as f:
        lines = f.readlines()

    # --- Split into optics vs particles ---
    optics_block = []
    particles_block = []
    in_particles = False
    for L in lines:
        if L.strip() == 'data_particles':
            in_particles = True
        if not in_particles:
            optics_block.append(L)
        else:
            particles_block.append(L)

    # --- Extract particle table header ---
    header_lines = []
    for L in particles_block:
        header_lines.append(L)
        if L.startswith('_rlnClassNumber'):
            break
    cols = [w.split()[0] for w in header_lines if w.startswith('_rln')]

    # --- Parse data rows into DataFrame ---
    data_rows = [
        L.strip().split()
        for L in particles_block[len(header_lines):]
        if L.strip()
    ]
    df = pd.DataFrame(data_rows, columns=cols)

    # --- Filter rows by micrograph basename ---
    mask = df['_rlnMicrographName'].str.endswith(tuple(keep))
    df_filt = df[mask]

    # --- Write out the filtered STAR file ---
    with open(output_star, 'w') as out:
        # 1) optics section
        out.writelines(optics_block)
        out.write('\n')
        # 2) particles section header
        out.write('data_particles\nloop_\n')
        out.writelines(header_lines)
        # 3) filtered rows
        for _, row in df_filt.iterrows():
            out.write(' '.join(row.values) + '\n')

    print(f"Filtered STAR written to: {output_star}")


    ### Val ####
    with open(file_path2, 'r') as f:
        keep2 = [line.strip() for line in f if line.strip()]


    # --- Read the entire STAR file ---
    with open(input_star, 'r') as f:
        lines = f.readlines()

    # --- Split into optics vs particles ---
    optics_block = []
    particles_block = []
    in_particles = False
    for L in lines:
        if L.strip() == 'data_particles':
            in_particles = True
        if not in_particles:
            optics_block.append(L)
        else:
            particles_block.append(L)

    # --- Extract particle table header ---
    header_lines = []
    for L in particles_block:
        header_lines.append(L)
        if L.startswith('_rlnClassNumber'):
            break
    cols = [w.split()[0] for w in header_lines if w.startswith('_rln')]

    # --- Parse data rows into DataFrame ---
    data_rows = [
        L.strip().split()
        for L in particles_block[len(header_lines):]
        if L.strip()
    ]
    df = pd.DataFrame(data_rows, columns=cols)

    # --- Filter rows by micrograph basename ---
    mask = df['_rlnMicrographName'].str.endswith(tuple(keep2))
    df_filt = df[mask]

    # --- Write out the filtered STAR file ---
    with open(output_star2, 'w') as out:
        # 1) optics section
        out.writelines(optics_block)
        out.write('\n')
        # 2) particles section header
        out.write('data_particles\nloop_\n')
        out.writelines(header_lines)
        # 3) filtered rows
        for _, row in df_filt.iterrows():
            out.write(' '.join(row.values) + '\n')

    print(f"Filtered STAR written to: {output_star2}")

    with open(file_path3, 'r') as f:
        keep3 = [line.strip() for line in f if line.strip()]


In [ ]:
if user:
    !cp /content/filtered_train.star {RESULT_DIR}/dataset/{EMPIAR_ID}/
    !cp /content/filtered_val.star {RESULT_DIR}/dataset/{EMPIAR_ID}/

---

### 🟪 Refining particle (by cryoSPARC).

> #### 🗒 Info
> As a developers you could directly used the particle stacks provided by the CryoPPP to obtain an accurate training mask. In this case, the step 1. use `particles_stack` directory under EMPIAR-ID and `particles_selected.star` from **`ground_truth`**.

> As a users, you could first run `00-B_Dataset preprocessing.ipynb` to split the dataset into train/val/test then you can reconstruct seperately from the validation set and training set to get different 3D masks.



> #### ⚠ Notice
>
> If you are using a STAR file other than those from **CryoPPP** (e.g., one that contains only particle coordinates), you must import **both** the particle and micrograph STAR files into **CryoSPARC**. After the import, run the [**Extract from Micrographs**](https://guide.cryosparc.com/processing-data/all-job-types-in-cryosparc/extraction/job-extract-from-micrographs) job to generate particle stacks for reconstruction. Remember to update the Job ID mentioned below so it points to **your** refinement job.

In [ ]:
# @title Set Cryosparc Parameter
PROJ_NAME             = "10017" # @param {type: "string"}
# @markdown ---
pixel_size            = 1.77    # @param {type: "number"}
accel_voltage         = 300     # @param {type: "number"}
spherical_aberration  = 2.7     # @param {type: "number"}
amplitude_contrast    = 0.1     # @param {type: "number"}
# @markdown ---
symmetry              = "D2"    # @param {type: "string"}
# @markdown ---
opt_gp_CTF            = False    # @param {type: "boolean"}
opt_prtc_defocus      = False    # @param {type: "boolean"}
# @markdown ---
star_train_dir = '/content/filtered_train.star'
star_val_dir = '/content/filtered_val.star'
particle_stacks_dir = f'{DATA_DIR}/EMPIAR-{EMPIAR_ID}/particles_stack/'

In [ ]:
def check_dir_in_gdrive(directory_path):
    if directory_path.startswith('/content/drive'):
        return True
    else :
        return False

is_gdrive = check_dir_in_gdrive(DATA_DIR)
if is_gdrive:
    # !cp {RESULT_DIR}/dataset/{EMPIAR_ID}/filtered_train.star  /content/filtered_train.star
    # !cp {RESULT_DIR}/dataset/{EMPIAR_ID}/filtered_val.star  /content/filtered_val.star
    !rsync -av {particle_stacks_dir}  /content/data/particles_stack
    particle_stacks_dir = "/content/data/particles_stack"

In [ ]:
# @title install `sparc-tools` package
!pip install cryosparc-tools

from cryosparc.tools import CryoSPARC

---
## Run Cryosparc

In [ ]:
# @title 0-0. Activate the cryosparc
start = True    # @param{type: "boolean"}
restart = False # @param{type: "boolean"}

sparcm = f'{SOFTWARE_DIR}/cryosparc/cryosparc_master/bin/cryosparcm'
if start == True:
    !{sparcm} start
if restart == True:
    !{sparcm} restart

In [ ]:
# @title View sparc status
status_view = False # @param{type: "boolean"}

if status_view == True:
    !{sparcm} status

In [ ]:
# @title Print `list_projects()` in sparc

result = subprocess.run(
    [sparcm, 'cli', 'list_projects()'],
    capture_output=True,
    text=True,
    check=True
    )

sparc_projects_list_str = result.stdout
# String to List
import ast
sparc_projects_list = ast.literal_eval(sparc_projects_list_str)

In [ ]:
# @title 0-1. Sparc Log in
hostname = socket.gethostname()

cs = CryoSPARC(
    license = LICENSE_ID,
    host = hostname,
    base_port = PORT,
    email = EMAIL,
    password = PASSWORD
)
assert cs.test_connection()

In [ ]:
# @title show all job types
show_all_job_types = False # @param{type: "boolean"}
if show_all_job_types == True:
    cs.print_job_types()

In [ ]:
# @title 0-2. Create a new sparc project
user_id = cs.cli.get_id_by_email(EMAIL)

new_project = cs.cli.create_empty_project(
    owner_user_id = user_id,
    title = PROJ_NAME,
    project_container_dir = EMPIAR_DIR
)

---
## J1

### Import particles

In [ ]:
# @title 1-1. Build job `import_particles`

# 1. Find your project
project = cs.find_project(new_project)

# 2. Build up workspace
workspace = project.create_workspace(title="workspace")

particles_job = workspace.create_job("import_particles")
print(f"Job Built {particles_job.uid}")

In [ ]:
# @title `import_particles` job spec show
particles_input_spec_show = False # @param{type: "boolean"}
if particles_input_spec_show == True:
    particles_job.print_input_spec()

particles_param_spec_show = False # @param{type: "boolean"}
if particles_param_spec_show == True:
    particles_job.print_param_spec()

In [ ]:
# @title 1-2. `import_particles` job execute

particles_job.set_param('particle_blob_path', particle_stacks_dir)
particles_job.set_param('particle_meta_path', star_train_dir)

particles_job.set_param('ignore_pose', True)
particles_job.set_param('remove_leading_uid', True)
particles_job.set_param('ignore_splits', True)

particles_job.set_param('accel_kv', accel_voltage)
particles_job.set_param('amp_contrast', amplitude_contrast)
particles_job.set_param('cs_mm', spherical_aberration)
particles_job.set_param('psize_A', pixel_size)

particles_job.set_param('enable_validation', True)  #strict check
particles_job.set_param('blob_exists', True)        #Particle raw data (blob)
particles_job.set_param('ctf_exists', True)         #Particle CTF parameters (ctf)

particles_job.queue()

In [ ]:
# @title ⚠️ Clear the job(particles), if any error happened

particles_job_clear = False # @param{type: "boolean"}

if particles_job_clear == True:
    particles_job.clear()

In [ ]:
# @title status print
status_print = True # @param{type: "boolean"}

if status_print == True:
    parent_job = project.find_job(particles_job.uid)
    print(f"Current Job Status: {parent_job.status}")

---
## J2

### Ab-Initio Reconstruction

In [ ]:
# @title 2-1. `ab-initio` job section create
abinit_job = workspace.create_job("homo_abinit")
print(f"Job Built {abinit_job.uid}")

# connect with extracct micrograph

abinit_job.connect(
    target_input="particles",
    source_job_uid=particles_job.uid,
    source_output="imported_particles",
)

In [ ]:
# @title `ab-initio` job spec show
abinit_input_spec_show = False # @param{type: "boolean"}
if abinit_input_spec_show == True:
    abinit_job.print_input_spec()

abinit_param_spec_show = False # @param{type: "boolean"}
if abinit_param_spec_show == True:
    abinit_job.print_param_spec()

In [ ]:
# @title 2-2. `ab-initio` job execute

# set param
abinit_job.set_param('abinit_symmetry', symmetry)

abinit_job.queue("default")
print(f" The Job ab-initio {abinit_job.uid} had sent into the queue！")

# wait for finisted
print(f"Waiting for job {abinit_job.uid} to finish...")
abinit_job.wait_for_done(error_on_incomplete=True)
print(f"Job {abinit_job.uid} has completed.")

In [ ]:
# @title ⚠️ Clear the job(ab-initio), if any error happened

abinit_job_clear = False # @param{type: "boolean"}

if abinit_job_clear == True:
    abinit_job.clear()

In [ ]:
# @title status print
status_print = True # @param{type: "boolean"}

if status_print == True:
    parent_job = project.find_job(abinit_job.uid)
    print(f"Current Job Status: {parent_job.status}")

---
## J3

### Non-Uniform Refinement


In [ ]:
# @title 3-1. `non-uniform_refinement` job section create
non_uni_job = workspace.create_job("nonuniform_refine_new")
print(f"Job Built {non_uni_job.uid}")

# connect with ab-initio
non_uni_job.connect(
    target_input="particles",
    source_job_uid=abinit_job.uid,
    source_output="particles_all_classes",
)

non_uni_job.connect(
    target_input="volume",
    source_job_uid=abinit_job.uid,
    source_output="volume_class_0",
)

In [ ]:
# @title `non-uniform_refinement` job spec show
non_uni_input_spec_show = False # @param{type: "boolean"}
if non_uni_input_spec_show == True:
    non_uni_job.print_input_spec()

non_uni_param_spec_show = False # @param{type: "boolean"}
if non_uni_param_spec_show == True:
    non_uni_job.print_param_spec()

In [ ]:
# @title 3-2. `non-uniform_refinement` job execute

# set param
non_uni_job.set_param('refine_symmetry', symmetry)
non_uni_job.set_param('refine_ctf_global_refine', opt_gp_CTF)
non_uni_job.set_param('refine_defocus_refine', opt_prtc_defocus)
non_uni_job.queue("default")
print(f" The Job non-uniform refinement {non_uni_job.uid} had sent into the queue！")

# wait for finisted
print(f"Waiting for job {non_uni_job.uid} to finish...")
non_uni_job.wait_for_done(error_on_incomplete=True)
print(f"Job {non_uni_job.uid} has completed.")

In [ ]:
# @title ⚠️ Clear the job( non-uniform refinement), if any error happened

non_uni_job_clear = False # @param{type: "boolean"}

if non_uni_job_clear == True:
    non_uni_job.clear()

In [ ]:
# @title status print
status_print = True # @param{type: "boolean"}

if status_print == True:
    parent_job = project.find_job(non_uni_job.uid)
    print(f"Current Job Status: {parent_job.status}")

---
## J4 (if user == True)

### Import particles

In [ ]:
# @title 4-1. Build job `import_particles`

if user == True:
    particles_job = workspace.create_job("import_particles")
    print(f"Job Built {particles_job.uid}")

In [ ]:
# @title `import_particles` job spec show

if user == True:
    particles_input_spec_show = False # @param{type: "boolean"}
    if particles_input_spec_show == True:
        particles_job.print_input_spec()

    particles_param_spec_show = False # @param{type: "boolean"}
    if particles_param_spec_show == True:
        particles_job.print_param_spec()

In [ ]:
# @title 4-2. `import_particles` job execute

particles_job.set_param('particle_blob_path', particle_stacks_dir)
particles_job.set_param('particle_meta_path', star_val_dir)

particles_job.set_param('ignore_pose', True)
particles_job.set_param('remove_leading_uid', True)
particles_job.set_param('ignore_splits', True)

particles_job.set_param('accel_kv', accel_voltage)
particles_job.set_param('amp_contrast', amplitude_contrast)
particles_job.set_param('cs_mm', spherical_aberration)
particles_job.set_param('psize_A', pixel_size)

particles_job.set_param('enable_validation', True)  #strict check
particles_job.set_param('blob_exists', True)        #Particle raw data (blob)
particles_job.set_param('ctf_exists', True)         #Particle CTF parameters (ctf)

particles_job.queue()

In [ ]:
# @title ⚠️ Clear the job(particles), if any error happened

if user == True:
    particles_job_clear = False # @param{type: "boolean"}

    if particles_job_clear == True:
        particles_job.clear()

In [ ]:
# @title status print

if user == True:
    status_print = True # @param{type: "boolean"}

    if status_print == True:
        parent_job = project.find_job(particles_job.uid)
        print(f"Current Job Status: {parent_job.status}")

---
## J5 (if user == True)

### Ab-Initio Reconstruction

In [ ]:
# @title 5-1. `ab-initio` job section create

if user == True:
    abinit_job = workspace.create_job("homo_abinit")
    print(f"Job Built {abinit_job.uid}")

    # connect with extracct micrograph

    abinit_job.connect(
        target_input="particles",
        source_job_uid=particles_job.uid,
        source_output="imported_particles",
    )


In [ ]:
# @title `ab-initio` job spec show

if user == True:
    abinit_input_spec_show = False # @param{type: "boolean"}
    if abinit_input_spec_show == True:
        abinit_job.print_input_spec()

    abinit_param_spec_show = False # @param{type: "boolean"}
    if abinit_param_spec_show == True:
        abinit_job.print_param_spec()

In [ ]:
# @title 5-2. `ab-initio` job execute

if user == True:
    # set param
    abinit_job.set_param('abinit_symmetry', symmetry)

    abinit_job.queue("default")
    print(f" The Job ab-initio {abinit_job.uid} had sent into the queue！")

    # wait for finisted
    print(f"Waiting for job {abinit_job.uid} to finish...")
    abinit_job.wait_for_done(error_on_incomplete=True)
    print(f"Job {abinit_job.uid} has completed.")

In [ ]:
# @title ⚠️ Clear the job(ab-initio), if any error happened

if user == True:
    abinit_job_clear = False # @param{type: "boolean"}

    if abinit_job_clear == True:
        abinit_job.clear()

In [ ]:
# @title status print

if user == True:
    status_print = True # @param{type: "boolean"}

    if status_print == True:
        parent_job = project.find_job(abinit_job.uid)
        print(f"Current Job Status: {parent_job.status}")

---
## J6 (if user == True)

### Non-Uniform Refinement


In [ ]:
# @title 6-1. `non-uniform_refinement` job section create

if user == True:
    non_uni_job = workspace.create_job("nonuniform_refine_new")
    print(f"Job Built {non_uni_job.uid}")

    # connect with ab-initio
    non_uni_job.connect(
        target_input="particles",
        source_job_uid=abinit_job.uid,
        source_output="particles_all_classes",
    )

    non_uni_job.connect(
        target_input="volume",
        source_job_uid=abinit_job.uid,
        source_output="volume_class_0",
    )

In [ ]:
# @title `non-uniform_refinement` job spec show

if user == True:
    non_uni_input_spec_show = False # @param{type: "boolean"}
    if non_uni_input_spec_show == True:
        non_uni_job.print_input_spec()

    non_uni_param_spec_show = False # @param{type: "boolean"}
    if non_uni_param_spec_show == True:
        non_uni_job.print_param_spec()

In [ ]:
# @title 6-2. `non-uniform_refinement` job execute

if user == True:
    # set param
    non_uni_job.set_param('refine_symmetry', symmetry)
    non_uni_job.set_param('refine_ctf_global_refine', opt_gp_CTF)
    non_uni_job.set_param('refine_defocus_refine', opt_prtc_defocus)
    non_uni_job.queue("default")
    print(f" The Job non-uniform refinement {non_uni_job.uid} had sent into the queue！")

    # wait for finisted
    print(f"Waiting for job {non_uni_job.uid} to finish...")
    non_uni_job.wait_for_done(error_on_incomplete=True)
    print(f"Job {non_uni_job.uid} has completed.")

In [ ]:
# @title ⚠️ Clear the job( non-uniform refinement), if any error happened

if user == True:
    non_uni_job_clear = False # @param{type: "boolean"}

    if non_uni_job_clear == True:
        non_uni_job.clear()

In [ ]:
# @title status print

if user == True:
    status_print = True # @param{type: "boolean"}

    if status_print == True:
        parent_job = project.find_job(non_uni_job.uid)
        print(f"Current Job Status: {parent_job.status}")

---

In [ ]:
# @title  { vertical-output: true, display-mode: "form" }
# @markdown **`EMPIAR DIR`**📁 should be chosen for project directory.
# @markdown The refinement should be implement in cryoSPARC by:
# @markdown 1. Import particle stack.
# @markdown 2. Build Ab-initio reconstruction.
# @markdown 3. Build non-uniform refinement.



if project_particles:
  # @markdown Assign `PROJ_NAME` using the name of the cryoSPARC project.
  particle_stack_Job = "J1" # @param {type:"string"}
  ab_initio_Job = "J2" # @param {type:"string"}
  refine_Job = "J3" # @param {type:"string"}
  if user:
    particle_stack_val_Job = "J4" # @param {type:"string"}
    ab_initio_val_Job = "J5" # @param {type:"string"}
    refine_val_Job = "J6" # @param {type:"string"}
  PROJECT_DIR = os.path.join(EMPIAR_DIR, f"CS-{PROJ_NAME}")
  RUN_REFINEMENT = False

  if os.path.exists(os.path.join(RESULT_DIR, f"refine_{EMPIAR_ID}.star")):
    shutil.copy(os.path.join(RESULT_DIR, f"refine_{EMPIAR_ID}.star"), PROCESSED_DIR)
    print(f"Using refinement result from {RESULT_DIR}")
  elif os.path.exists(os.path.join(PROCESSED_DIR, "refinement_result.star")):
    print("File 'refinement_result.star' already exists.")
  elif os.path.isdir(PROJECT_DIR):
    print(f"Refinement finished.")
    RUN_REFINEMENT = True
  else:
    try:
      from pyngrok import ngrok, conf
      import getpass

      print("Enter your authtoken, which can be copied from https://dashboard.ngrok.com/")
      conf.get_default().auth_token = getpass.getpass()
      # Setup a tunnel to the port 61000
      public_url = ngrok.connect(61000)
      print(public_url)
    except:
      pass
    raise BaseException(
        "Open cryoSPARC for refinement. " +\
        f"Remember to name the project '{PROJ_NAME}'.\n" +\
        "Rerun this cell after refinement is done."
    )

In [ ]:
PROJ_NAME

In [ ]:
PROJECT_DIR

In [ ]:
# @title  { vertical-output: true, display-mode: "form" }
# @markdown Backup entire cryoSPARC porject to **`RESULT DIR`**📁.

backup = True # @param {type:"boolean"}

if project_particles and backup:
  tmp_path = os.path.join(RESULT_DIR, f"CS-{PROJ_NAME}")
  if os.path.isdir(tmp_path):
    print(f"File exists: '{tmp_path}'")
  else:
    #os.mkdir(tmp_path)
    print(f"Copy 'CS-{PROJ_NAME}' from '{EMPIAR_DIR}' to '{RESULT_DIR}'.")
    shutil.copytree(PROJECT_DIR, tmp_path)

### 🟪 Convert refinement result from cryoSPARC form to relion form (by pyem).

In [ ]:
# @title  { vertical-output: true, display-mode: "form" }
# PROJECT_DIR = tmp_path

if project_particles:
  if os.path.exists(f"{PROCESSED_DIR}/refinement_result.star"):
    print("File 'refinement_result.star' already exists.")
  elif not os.path.isdir(f"{PROJECT_DIR}"):
    raise FileNotFoundError(f"No refinement result inside '{PROJECT_DIR}'.")
  else:
    if user:
        with open(f"{PROJECT_DIR}/{refine_Job}/{refine_Job}_particles.csg", 'r') as f:
            refine_particles = yaml.safe_load(f)
        refinement_ctf = refine_particles['results']['ctf']['metafile'][1:]
        refinement_blob = refine_particles['results']['blob']['metafile'][1:]
        # csparc2star --inverty?
        !csparc2star.py \
                {PROJECT_DIR}/{refine_Job}/{refinement_ctf} \
                {PROJECT_DIR}/{refine_Job}/{refinement_blob} \
                {PROCESSED_DIR}/refinement_result_train.star
        shutil.copy(f"{PROCESSED_DIR}/refinement_result_train.star", f"{RESULT_DIR}/refine_{EMPIAR_ID}_train.star")

        with open(f"{PROJECT_DIR}/{refine_val_Job}/{refine_val_Job}_particles.csg", 'r') as f:
            refine_particles = yaml.safe_load(f)
        refinement_ctf2 = refine_particles['results']['ctf']['metafile'][1:]
        refinement_blob2 = refine_particles['results']['blob']['metafile'][1:]
        # csparc2star --inverty?
        !csparc2star.py \
                {PROJECT_DIR}/{refine_val_Job}/{refinement_ctf2} \
                {PROJECT_DIR}/{refine_val_Job}/{refinement_blob2} \
                {PROCESSED_DIR}/refinement_result_val.star
        shutil.copy(f"{PROCESSED_DIR}/refinement_result_val.star", f"{RESULT_DIR}/refine_{EMPIAR_ID}_val.star")
    else:
        with open(f"{PROJECT_DIR}/{refine_Job}/{refine_Job}_particles.csg", 'r') as f:
            refine_particles = yaml.safe_load(f)
        refinement_ctf = refine_particles['results']['ctf']['metafile'][1:]
        refinement_blob = refine_particles['results']['blob']['metafile'][1:]
        # csparc2star --inverty?
        !csparc2star.py \
                {PROJECT_DIR}/{refine_Job}/{refinement_ctf} \
                {PROJECT_DIR}/{refine_Job}/{refinement_blob} \
                {PROCESSED_DIR}/refinement_result.star
        shutil.copy(f"{PROCESSED_DIR}/refinement_result.star", f"{RESULT_DIR}/refine_{EMPIAR_ID}.star")

### 🟪 Project particle (by relion).

In [ ]:
# @title  { display-mode: "form" }

project_mask = True # @param {type:"boolean"}

if project_particles:
#   if not os.path.exists(f"{PROCESSED_DIR}/refinement_result.star"):
#     raise FileNotFoundError(
#         f'refinement_result.star was not found inside {PROCESSED_DIR}.\n' +\
#         'Run the cell above to refine the particle first".')

  if user:
    with open(f"{PROJECT_DIR}/{refine_Job}/{refine_Job}_volume.csg", 'r') as f:
        refine_volume_csg = yaml.safe_load(f)
    if project_mask:
        volume_map = refine_volume_csg['results']['mask_refine']['metafile'][1:]
    else:
        volume_map = refine_volume_csg['results']['map']['metafile'][1:]
    volume_map = volume_map.replace('.cs', '.mrc')
    !relion_project \
      --i {PROJECT_DIR}/{refine_Job}/{volume_map} \
      --o {PROCESSED_DIR}/proj_{EMPIAR_ID}_train \
      --ang {PROCESSED_DIR}/refinement_result_train.star
    !du -sh {PROCESSED_DIR}/proj_{EMPIAR_ID}_train.mrcs

    with open(f"{PROJECT_DIR}/{refine_val_Job}/{refine_val_Job}_volume.csg", 'r') as f:
        refine_volume_csg = yaml.safe_load(f)
    if project_mask:
        volume_map = refine_volume_csg['results']['mask_refine']['metafile'][1:]
    else:
        volume_map = refine_volume_csg['results']['map']['metafile'][1:]
    volume_map = volume_map.replace('.cs', '.mrc')
    !relion_project \
      --i {PROJECT_DIR}/{refine_val_Job}/{volume_map} \
      --o {PROCESSED_DIR}/proj_{EMPIAR_ID}_val \
      --ang {PROCESSED_DIR}/refinement_result_val.star
    !du -sh {PROCESSED_DIR}/proj_{EMPIAR_ID}_val.mrcs
  else:
    with open(f"{PROJECT_DIR}/{refine_Job}/{refine_Job}_volume.csg", 'r') as f:
        refine_volume_csg = yaml.safe_load(f)
    if project_mask:
        volume_map = refine_volume_csg['results']['mask_refine']['metafile'][1:]
    else:
        volume_map = refine_volume_csg['results']['map']['metafile'][1:]
    volume_map = volume_map.replace('.cs', '.mrc')
    !relion_project \
      --i {PROJECT_DIR}/{refine_Job}/{volume_map} \
      --o {PROCESSED_DIR}/proj_{EMPIAR_ID} \
      --ang {PROCESSED_DIR}/refinement_result.star
    !du -sh {PROCESSED_DIR}/proj_{EMPIAR_ID}.mrcs

In [ ]:
# @markdown Save particle porjection to **`RESULT DIR`**📁. { display-mode: "form" }

to_result_dir = True # @param {type:"boolean"}

if project_particles and to_result_dir:
  if user:
    shutil.copy(f"{PROCESSED_DIR}/proj_{EMPIAR_ID}_train.star", f"{RESULT_DIR}/dataset/{EMPIAR_ID}/proj_{EMPIAR_ID}_train.star")
    shutil.copy(f"{PROCESSED_DIR}/proj_{EMPIAR_ID}_train.mrcs", f"{RESULT_DIR}/dataset/{EMPIAR_ID}/proj_{EMPIAR_ID}_train.mrcs")
    shutil.copy(f"{PROCESSED_DIR}/proj_{EMPIAR_ID}_val.star", f"{RESULT_DIR}/dataset/{EMPIAR_ID}/proj_{EMPIAR_ID}_val.star")
    shutil.copy(f"{PROCESSED_DIR}/proj_{EMPIAR_ID}_val.mrcs", f"{RESULT_DIR}/dataset/{EMPIAR_ID}/proj_{EMPIAR_ID}_val.mrcs")
  else:
    shutil.copy(f"{PROCESSED_DIR}/proj_{EMPIAR_ID}.star", f"{RESULT_DIR}/dataset/{EMPIAR_ID}/proj_{EMPIAR_ID}.star")
    shutil.copy(f"{PROCESSED_DIR}/proj_{EMPIAR_ID}.mrcs", f"{RESULT_DIR}/dataset/{EMPIAR_ID}/proj_{EMPIAR_ID}.mrcs")

## ⭐ Segmentation

### ✅ Preparation

In [ ]:
# @markdown Import packages.

import matplotlib
from scipy import ndimage as ndi
from skimage import filters, segmentation, morphology
from skimage.segmentation import chan_vese, clear_border
from skimage.filters import thresholding
from skimage import transform

In [ ]:
# @markdown Define function. { display-mode: "form" }

colors = [(1,0,0,c) for c in np.linspace(0,1,100)]
red_transp = matplotlib.colors.LinearSegmentedColormap.from_list('mycmap', colors)

def simple_micrograph_preprocessing(micrograph):
  micrograph_copy = micrograph.copy()
  micrograph_copy = (micrograph_copy-micrograph.mean()+2.5*micrograph.std())/5/micrograph.std()
  micrograph_copy[micrograph_copy<0]=0
  micrograph_copy[micrograph_copy>1]=1
  return micrograph_copy

def segment(image):
  threshold = thresholding.threshold_li(image)
  image_seg = image > threshold
  image_seg = segmentation.clear_border(image_seg)
  image_seg = morphology.remove_small_objects(image_seg, 64)
  return image_seg

def segment_fill(image):
  image_seg = image > thresholding.threshold_li(image)
  image_seg = segmentation.clear_border(image_seg)
  image_seg = ndi.binary_fill_holes(image_seg)
  return image_seg

def segment_yen(image):
  image_seg = image > thresholding.threshold_yen(image)
  image_seg = ndi.binary_fill_holes(image_seg)
  image_seg = morphology.remove_small_objects(image_seg, 64)
  return image_seg

def segment_triangle(image):
  image_seg = image > thresholding.threshold_triangle(image)
  image_seg = ndi.binary_fill_holes(image_seg)
  image_seg = morphology.remove_small_objects(image_seg, 64)
  return image_seg


In [ ]:
# @title  { display-mode: "form" }
# @markdown Select micrograph.

name = 'Falcon_2012_06_13-00_19_28_0' # @param {type:"string"}
# f"{PROCESSED_DIR}/{name}.mrcs"
with mrcfile.open(f"{EMPIAR_DIR}/micrographs/{name}.mrc") as mrcs:
  micrograph = mrcs.data

In [ ]:
# @markdown Read location. { display-mode: "form" }

y_size = micrograph.shape[0]
print(y_size)
# labeled_particles = pd.read_csv(f"{PROCESSED_DIR}/particles.txt", sep='\t')
# labeled_particles = starfile.read(f"{PROCESSED_DIR}/selected.star")
if user:
    labeled_particles = starfile.read(f"/content/filtered_train.star")['particles']
    labeled_particles = labeled_particles[['rlnMicrographName', 'rlnCoordinateX', 'rlnCoordinateY']]
    labeled_particles.columns = pd.Index(['image_name', 'x_coord', 'y_coord'])
    labeled_particles['image_name'] = labeled_particles['image_name'].apply(get_basename_with_uid_removed)
    labeled_particles['image_name'] = labeled_particles['image_name'].apply(lambda s: s.split(".")[0])
    labeled_particles['y_coord'] = y_size - labeled_particles['y_coord']

    labeled_val_particles = starfile.read(f"/content/filtered_val.star")['particles']
    labeled_val_particles = labeled_val_particles[['rlnMicrographName', 'rlnCoordinateX', 'rlnCoordinateY']]
    labeled_val_particles.columns = pd.Index(['image_name', 'x_coord', 'y_coord'])
    labeled_val_particles['image_name'] = labeled_val_particles['image_name'].apply(get_basename_with_uid_removed)
    labeled_val_particles['image_name'] = labeled_val_particles['image_name'].apply(lambda s: s.split(".")[0])
    labeled_val_particles['y_coord'] = y_size - labeled_val_particles['y_coord']
else:
    labeled_particles = starfile.read(f"{EMPIAR_DIR}/ground_truth/empiar-{EMPIAR_ID}_particles_selected.star")['particles']
    labeled_particles = labeled_particles[['rlnMicrographName', 'rlnCoordinateX', 'rlnCoordinateY']]
    labeled_particles.columns = pd.Index(['image_name', 'x_coord', 'y_coord'])
    labeled_particles['image_name'] = labeled_particles['image_name'].apply(get_basename_with_uid_removed)
    labeled_particles['image_name'] = labeled_particles['image_name'].apply(lambda s: s.split(".")[0])
    labeled_particles['y_coord'] = y_size - labeled_particles['y_coord']
labeled_particles

In [ ]:
# @markdown Read particles. { display-mode: "form" }
import mrcfile
if user:
    with mrcfile.open(f"{PROCESSED_DIR}/proj_{EMPIAR_ID}_train.mrcs") as mrcs:
        particles = mrcs.data
    print("particles:", particles.shape)
    image_names = labeled_particles['image_name'].unique()
    pd.DataFrame({'image_name':image_names[0:5]})

    with mrcfile.open(f"{PROCESSED_DIR}/proj_{EMPIAR_ID}_val.mrcs") as mrcs:
        particles2 = mrcs.data
    print("particles:", particles2.shape)
    image_names_val = labeled_val_particles['image_name'].unique()
    pd.DataFrame({'image_name':image_names_val[0:5]})
else:
    with mrcfile.open(f"{PROCESSED_DIR}/proj_{EMPIAR_ID}.mrcs") as mrcs:
        particles = mrcs.data
    print("particles:", particles.shape)
    image_names = labeled_particles['image_name'].unique()
    pd.DataFrame({'image_name':image_names[0:5]})

### Using cricle mask

In [ ]:
# @title { display-mode: "form" }
# @markdown If the volume appears to have low resolution,
# @markdown you may directly specify the radius and apply a circular mask.
# @markdown Please proceed to the **Circle Mask** section to generate the reference mask.

circle_mask = False # @param {type:"boolean"}
radius = 64 # @param {type:"integer"}
# Parameters
n_images = len(labeled_particles)+len(labeled_val_particles)
_, H, W = particles.shape

# 1) Create coordinate grids centered at (H/2, W/2)
y = np.arange(H) - (H - 1) / 2
x = np.arange(W) - (W - 1) / 2
yy, xx = np.meshgrid(y, x, indexing='ij')

# 2) Compute distance from center for each pixel
dist = np.sqrt(xx**2 + yy**2)

# 3) Build the single circular mask: inside circle = 255 (white), outside = 0 (black)
mask = np.zeros((H, W), dtype=np.uint8)
mask[dist <= radius] = 255

# 4) Stack/repeat this mask into an array of shape (9600, 256, 256)
images = np.broadcast_to(mask, (n_images, H, W)).copy()
# — or equivalently:
# images = np.tile(mask, (n_images, 1, 1))

print(images.shape)  # (9600, 256, 256)
print(images.dtype)  # uint8, values 0 or 255

### ⏭ Segmentation preparation.

In [ ]:
# @title  { display-mode: "form" }
# @markdown Plot particle.

radius = 64 # @param {type:"integer"}
number_of_particle = particles.shape
idx = 2 # @param {type:"integer"}
image = particles[idx]
plt.imshow(image, 'gray')
plt.title(f"proj_{EMPIAR_ID}_{idx}")
plt.axis('off')
plt.show()

#### Test segmentation methods

In [ ]:
locations = labeled_particles[labeled_particles['image_name'] == name]

In [ ]:
# @markdown Micrograph intensity histogram.

plt.subplots(figsize=(6, 4))
plt.hist(micrograph.ravel(), bins=1000)
plt.title(f"Intensity histogram of {name+'.mrc'}")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches

# Assume `simple_micrograph_preprocessing` and `locations` are defined elsewhere
_, ax = plt.subplots(figsize=(12, 12))
micrograph_copy = simple_micrograph_preprocessing(micrograph)
ax.imshow(micrograph_copy, cmap='gray')

# Set the limits to zoom in
ax.set_xlim(300, 1300)
ax.set_ylim(2100, 1100)  # Note: the y-axis in images goes from top to bottom

# Adding the circle patches
#indx = locations.index[0]
#x, y = locations.loc[indx][['x_coord', 'y_coord']]
#c = matplotlib.patches.Circle((x, y), radius=64, fill=False, color='b')
#ax.add_patch(c)
for indx in locations.index[1:]:
    x, y = locations.loc[indx][['x_coord', 'y_coord']]
    c = matplotlib.patches.Circle((x, y), radius=radius, fill=False, color='r', linewidth=5)
    ax.add_patch(c)

plt.show()

In [ ]:
# @markdown Particles view. { display-mode: "form" }

_, ax = plt.subplots(figsize=(12, 12))
micrograph_copy = simple_micrograph_preprocessing(micrograph)
ax.imshow(micrograph_copy, cmap='gray')

indx = locations.index[0]
x, y = locations.loc[indx][['x_coord', 'y_coord']]
c = matplotlib.patches.Circle((x, y), radius=radius, fill=False, color='b')
ax.add_patch(c)
for indx in locations.index[1:]:
  x, y = locations.loc[indx][['x_coord', 'y_coord']]
  c = matplotlib.patches.Circle((x, y), radius=radius, fill=False, color='r')
  ax.add_patch(c)
plt.show()

In [ ]:
if user:
    df = starfile.read(f"{EMPIAR_DIR}/processed/refinement_result_train.star")
    df_optics2, df_particles2 = df['optics'].copy(), df['particles'].copy()
    df = starfile.read(f"{EMPIAR_DIR}/processed/refinement_result_val.star")
    df_optics2_val, df_particles2_val = df['optics'].copy(), df['particles'].copy()
else:
    df = starfile.read(f"{EMPIAR_DIR}/processed/refinement_result.star")
    df_optics2, df_particles2 = df['optics'].copy(), df['particles'].copy()

In [ ]:
pd.DataFrame(df_particles2.loc[indx])

In [ ]:
locations.loc[indx]

In [ ]:
# @title  { display-mode: "form" }
# @markdown Segmentation demonstration.

pixel_size = 1.77 # @param {type:"number"}
idx = 53 # @param {type:"integer"}
indx = locations.index[idx]
x, y = locations.loc[indx][['x_coord', 'y_coord']]
image = particles[indx].copy()
tform = transform.EuclideanTransform(
   rotation = 0,
   translation = (df_particles2.loc[indx]['rlnOriginXAngst']/pixel_size, df_particles2.loc[indx]['rlnOriginYAngst']/pixel_size)
   )
tf_img = transform.warp(image, tform.inverse)
image_seg = segment(tf_img)

# fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
ax = axes.flatten()

ax[0].imshow(micrograph_copy[y-128:y+128,x-128:x+128], cmap="gray")
#ax[0].imshow(image_seg, cmap="gray", alpha=0.5)
ax[0].set_axis_off()
ax[0].set_title(f"Particle {indx} on micrograph.", fontsize=12)
c = matplotlib.patches.Circle((x, y), radius=64, fill=False, color='r')
ax[0].add_patch(c)

ax[1].imshow(micrograph_copy[y-128:y+128,x-128:x+128], cmap="gray")
#ax[1].imshow(tf_img, cmap="gray")
ax[1].imshow(image_seg, cmap="gray", alpha=0.3)
ax[1].set_axis_off()
ax[1].set_title(f"Particle {indx}.", fontsize=12)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
ax = axes.flatten()

ax[0].imshow(tf_img, cmap="gray")
ax[0].imshow(image_seg, cmap="gray", alpha=0.5)
ax[0].set_axis_off()
ax[0].set_title(f"Particle {indx} on micrograph.", fontsize=12)
c = matplotlib.patches.Circle((x, y), radius=77, fill=False, color='r')
ax[0].add_patch(c)

#ax[1].imshow(micrograph_copy[y-128:y+128,x-128:x+128], cmap="gray")
ax[1].imshow(image_seg, cmap="gray")
#ax[1].imshow(image_seg, cmap="gray", alpha=0.3)
ax[1].set_axis_off()
ax[1].set_title(f"Particle {indx}.", fontsize=12)
plt.show()

In [ ]:
# @title  { display-mode: "form" }
# @markdown Particles segment result.

diameter = 128 # @param {type:"integer"}
pixel_size = 1.77 # @param {type:"number"}
micrograph_copy = simple_micrograph_preprocessing(micrograph)
micrograph_segment = np.zeros_like(micrograph, dtype=bool) # .astype(bool)
micrograph_segment_fill = np.zeros_like(micrograph, dtype=bool)
micrograph_segment_triangle = np.zeros_like(micrograph, dtype=bool)

for indx in locations.index:
  x, y = locations.loc[indx][['x_coord', 'y_coord']]
  image = particles[indx].copy()
  tform = transform.EuclideanTransform(
    rotation = 0,
    translation = (df_particles2.loc[indx]['rlnOriginXAngst']/pixel_size, df_particles2.loc[indx]['rlnOriginYAngst']/pixel_size)
    )
  tf_img = transform.warp(image, tform.inverse)
  image_seg = segment(tf_img)
  micrograph_segment[y-diameter:y+diameter,x-diameter:x+diameter] += image_seg
  micrograph_segment_fill[y-diameter:y+diameter,x-diameter:x+diameter] += segment_fill(image)
  micrograph_segment_triangle[y-diameter:y+diameter,x-diameter:x+diameter] += segment_triangle(image)

fig, axes = plt.subplots(2, 2, figsize=(16, 16))
ax = axes.flatten()

ax[0].imshow(micrograph_copy, cmap='gray')
ax[0].set_axis_off()
ax[0].set_title("Micrographs", fontsize=12)

ax[1].imshow(micrograph_segment, cmap='inferno')
ax[1].set_axis_off()
ax[1].set_title("Segmenting result using Li's threshold and removing small objects per particles", fontsize=12)

ax[2].imshow(micrograph_segment_fill, cmap='inferno')
ax[2].set_axis_off()
ax[2].set_title("Segment result using Li's threshold and filling holes per particles", fontsize=12)

ax[3].imshow(micrograph_segment_triangle, cmap='inferno')
ax[3].set_axis_off()
ax[3].set_title("Segmenting result using Triangle threshold and removing small objects per particles", fontsize=12)

fig.tight_layout()
plt.show()

In [ ]:
# @title  { display-mode: "form" }
# @markdown Check whether small objects have been created.

micrograph_segment_clear = morphology.remove_small_objects(micrograph_segment, 2048)
micrograph_segment_removed = np.logical_xor(micrograph_segment, micrograph_segment_clear)

micrograph_segment_fill_clear = morphology.remove_small_objects(micrograph_segment_fill, 2048)
micrograph_segment_fill_removed = np.logical_xor(micrograph_segment_fill, micrograph_segment_fill_clear)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
ax = axes.flatten()

ax[0].imshow(micrograph_segment_clear, cmap='inferno')
ax[0].set_axis_off()
ax[0].set_title("Result that small object removed after combining all segmentation", fontsize=12)

ax[1].imshow(micrograph_segment_removed, cmap='inferno')
ax[1].set_axis_off()
ax[1].set_title("The small object removed after combining all segmentation", fontsize=12)

fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
ax = axes.flatten()

ax[0].imshow(micrograph_segment_fill_clear, cmap='inferno')
ax[0].set_axis_off()
ax[0].set_title("Result that small object removed after combining all segmentation (fill)", fontsize=12)

ax[1].imshow(micrograph_segment_fill_removed, cmap='inferno')
ax[1].set_axis_off()
ax[1].set_title("The small object removed after combining all segmentation (fill)", fontsize=12)

fig.tight_layout()
plt.show()

print("Segmentation created small objects:", micrograph_segment_removed.any())
print("Segmentation (fill) created small objects:", micrograph_segment_fill_removed.any())

In [ ]:
# @markdown Particles

_,ax = plt.subplots(figsize=(12, 12))

ax.imshow(micrograph_copy, cmap='gray')
ax.imshow(micrograph_segment, cmap='inferno', alpha=0.4)

indx = locations.index[0]
x, y = locations.loc[indx][['x_coord', 'y_coord']]
c = matplotlib.patches.Circle((x, y), radius=radius, fill=False, color='b')
ax.add_patch(c)
for indx in locations.index[1:]:
  x, y = locations.loc[indx][['x_coord', 'y_coord']]
  c = matplotlib.patches.Circle((x, y), radius=radius, fill=False, color='r')
  ax.add_patch(c)
plt.show()

Just using minimum cross entropy thresholding is enough.

### ✅ Segmenting all micrograph

In [ ]:
# @markdown Set directory.
# @markdown If any of the following directory is not given, the image will not be saved.
MICROGRAPH_DIR = "micrographs_np" # @param {type:"string"}
PROCESSED_MICROGRAPH_DIR = "processed_micrographs_np" # @param {type:"string"}
GROUND_TRUTH_DIR = "micrographs_ground_np" # @param {type:"string"}

if not os.path.exists(MICROGRAPH_DIR):
  !mkdir {RESULT_DIR}/dataset/{EMPIAR_ID}/{MICROGRAPH_DIR}

if not os.path.exists(PROCESSED_MICROGRAPH_DIR):
  !mkdir {RESULT_DIR}/dataset/{EMPIAR_ID}/{PROCESSED_MICROGRAPH_DIR}

if not os.path.exists(GROUND_TRUTH_DIR):
  !mkdir {RESULT_DIR}/dataset/{EMPIAR_ID}/{GROUND_TRUTH_DIR}

In [ ]:
labeled_particles

In [ ]:
%%capture --no-display
# @markdown Run segmentation for all micrographs.

pixel_size = 1.77 # @param {type:"number"}
diameter = 128 # @param {type:"integer"}
plot = True # @param {type:"boolean"}
plot_processed = True # @param {type:"boolean"}

if not os.path.exists(MICROGRAPH_DIR):
  !mkdir {RESULT_DIR}/dataset/{EMPIAR_ID}/{MICROGRAPH_DIR}

if not os.path.exists(PROCESSED_MICROGRAPH_DIR):
  !mkdir {RESULT_DIR}/dataset/{EMPIAR_ID}/{PROCESSED_MICROGRAPH_DIR}

if not os.path.exists(GROUND_TRUTH_DIR):
  !mkdir {RESULT_DIR}/dataset/{EMPIAR_ID}/{GROUND_TRUTH_DIR}

if user:
    image_names = keep
    for name in image_names:
        name = name[:-4]
        with mrcfile.open(f"{EMPIAR_DIR}/micrographs/{name}.mrc") as mrcs:
            micrograph = mrcs.data
        locations = labeled_particles[labeled_particles['image_name'] == name]
        micrograph_segment = np.zeros_like(micrograph, dtype=bool)

        for indx in locations.index:
            x, y = locations.loc[indx][['x_coord', 'y_coord']]
            image = particles[indx].copy()
            tform = transform.EuclideanTransform(
                rotation = 0,
                translation = (df_particles2.loc[indx]['rlnOriginXAngst']/pixel_size, df_particles2.loc[indx]['rlnOriginYAngst']/pixel_size)
                )
            tf_img = transform.warp(image, tform.inverse)
            image_seg = segment(tf_img)
            micrograph_segment[y-diameter:y+diameter,x-diameter:x+diameter] += image_seg

        if MICROGRAPH_DIR:
            img = Image.fromarray(np.uint8(255 * micrograph))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{MICROGRAPH_DIR}/{name}.png")

        if PROCESSED_MICROGRAPH_DIR:
            with mrcfile.open(f"{PROCESSED_DIR}/micrographs/{name}.mrc", permissive=True) as mrcs:
                processed_micrograph = mrcs.data
            img = Image.fromarray(np.uint8(255 * processed_micrograph))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{PROCESSED_MICROGRAPH_DIR}/{name}.png")

        if GROUND_TRUTH_DIR:
            img = Image.fromarray(np.uint8(255 * micrograph_segment))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{GROUND_TRUTH_DIR}/{name}.png")

        if plot:
            _, ax = plt.subplots(figsize=(12, 12))

            if plot_processed:
                ax.imshow(simple_micrograph_preprocessing(processed_micrograph), cmap='gray')
            else:
                ax.imshow(simple_micrograph_preprocessing(micrograph), cmap='gray')
            ax.imshow(micrograph_segment, cmap='inferno', alpha=0.4)

            for indx in locations.index:
                x, y = locations.loc[indx][['x_coord', 'y_coord']]
                c = matplotlib.patches.Circle((x, y), radius=radius, fill=False, color='r')
                ax.add_patch(c)

            plt.show()

    image_names = keep2
    for name in image_names:
        name = name[:-4]
        with mrcfile.open(f"{EMPIAR_DIR}/micrographs/{name}.mrc") as mrcs:
            micrograph = mrcs.data
        locations = labeled_val_particles[labeled_val_particles['image_name'] == name]
        micrograph_segment = np.zeros_like(micrograph, dtype=bool)

        for indx in locations.index:
            x, y = locations.loc[indx][['x_coord', 'y_coord']]
            image = particles2[indx].copy()
            tform = transform.EuclideanTransform(
                rotation = 0,
                translation = (df_particles2_val.loc[indx]['rlnOriginXAngst']/pixel_size, df_particles2_val.loc[indx]['rlnOriginYAngst']/pixel_size)
                )
            tf_img = transform.warp(image, tform.inverse)
            image_seg = segment(tf_img)
            micrograph_segment[y-diameter:y+diameter,x-diameter:x+diameter] += image_seg

        if MICROGRAPH_DIR:
            img = Image.fromarray(np.uint8(255 * micrograph))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{MICROGRAPH_DIR}/{name}.png")

        if PROCESSED_MICROGRAPH_DIR:
            with mrcfile.open(f"{PROCESSED_DIR}/micrographs/{name}.mrc", permissive=True) as mrcs:
                processed_micrograph = mrcs.data
            img = Image.fromarray(np.uint8(255 * processed_micrograph))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{PROCESSED_MICROGRAPH_DIR}/{name}.png")

        if GROUND_TRUTH_DIR:
            img = Image.fromarray(np.uint8(255 * micrograph_segment))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{GROUND_TRUTH_DIR}/{name}.png")

        if plot:
            _, ax = plt.subplots(figsize=(12, 12))

            if plot_processed:
                ax.imshow(simple_micrograph_preprocessing(processed_micrograph), cmap='gray')
            else:
                ax.imshow(simple_micrograph_preprocessing(micrograph), cmap='gray')
            ax.imshow(micrograph_segment, cmap='inferno', alpha=0.4)

            for indx in locations.index:
                x, y = locations.loc[indx][['x_coord', 'y_coord']]
                c = matplotlib.patches.Circle((x, y), radius=radius, fill=False, color='r')
                ax.add_patch(c)

            plt.show()
    image_names = keep3
    for name in image_names:
        name = name[:-4]
        if MICROGRAPH_DIR:
            img = Image.fromarray(np.uint8(255 * micrograph))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{MICROGRAPH_DIR}/{name}.png")

        if PROCESSED_MICROGRAPH_DIR:
            with mrcfile.open(f"{PROCESSED_DIR}/micrographs/{name}.mrc", permissive=True) as mrcs:
                processed_micrograph = mrcs.data
            img = Image.fromarray(np.uint8(255 * processed_micrograph))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{PROCESSED_MICROGRAPH_DIR}/{name}.png")
else:
    for name in image_names:
        with mrcfile.open(f"{EMPIAR_DIR}/micrographs/{name}.mrc") as mrcs:
            micrograph = mrcs.data
        locations = labeled_particles[labeled_particles['image_name'] == name]
        micrograph_segment = np.zeros_like(micrograph, dtype=bool)

        for indx in locations.index:
            x, y = locations.loc[indx][['x_coord', 'y_coord']]
            image = particles[indx].copy()
            tform = transform.EuclideanTransform(
                rotation = 0,
                translation = (df_particles2.loc[indx]['rlnOriginXAngst']/pixel_size, df_particles2.loc[indx]['rlnOriginYAngst']/pixel_size)
                )
            tf_img = transform.warp(image, tform.inverse)
            image_seg = segment(tf_img)
            micrograph_segment[y-diameter:y+diameter,x-diameter:x+diameter] += image_seg

        if MICROGRAPH_DIR:
            img = Image.fromarray(np.uint8(255 * micrograph))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{MICROGRAPH_DIR}/{name}.png")

        if PROCESSED_MICROGRAPH_DIR:
            with mrcfile.open(f"{PROCESSED_DIR}/micrographs/{name}.mrc", permissive=True) as mrcs:
                processed_micrograph = mrcs.data
            img = Image.fromarray(np.uint8(255 * processed_micrograph))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{PROCESSED_MICROGRAPH_DIR}/{name}.png")

        if GROUND_TRUTH_DIR:
            img = Image.fromarray(np.uint8(255 * micrograph_segment))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{GROUND_TRUTH_DIR}/{name}.png")

        if plot:
            _, ax = plt.subplots(figsize=(12, 12))

            if plot_processed:
                ax.imshow(simple_micrograph_preprocessing(processed_micrograph), cmap='gray')
            else:
                ax.imshow(simple_micrograph_preprocessing(micrograph), cmap='gray')
            ax.imshow(micrograph_segment, cmap='inferno', alpha=0.4)

            for indx in locations.index:
                x, y = locations.loc[indx][['x_coord', 'y_coord']]
                c = matplotlib.patches.Circle((x, y), radius=radius, fill=False, color='r')
                ax.add_patch(c)

            plt.show()

## Circle Mask

In [ ]:
%%capture --no-display
# @markdown Run segmentation for all micrographs.
"""
pixel_size = 1.77 # @param {type:"number"}
diameter = 128 # @param {type:"integer"}
plot = True # @param {type:"boolean"}
plot_processed = True # @param {type:"boolean"}

if not os.path.exists(MICROGRAPH_DIR):
  !mkdir {RESULT_DIR}/dataset/{EMPIAR_ID}/{MICROGRAPH_DIR}

if not os.path.exists(PROCESSED_MICROGRAPH_DIR):
  !mkdir {RESULT_DIR}/dataset/{EMPIAR_ID}/{PROCESSED_MICROGRAPH_DIR}

if not os.path.exists(GROUND_TRUTH_DIR):
  !mkdir {RESULT_DIR}/dataset/{EMPIAR_ID}/{GROUND_TRUTH_DIR}

particles = images
if user:
    image_names = keep
    for name in image_names:
        name = name[:-4]
        with mrcfile.open(f"{EMPIAR_DIR}/micrographs/{name}.mrc") as mrcs:
            micrograph = mrcs.data
        locations = labeled_particles[labeled_particles['image_name'] == name]
        micrograph_segment = np.zeros_like(micrograph, dtype=bool)

        for indx in locations.index:
            x, y = locations.loc[indx][['x_coord', 'y_coord']]
            image = particles[indx].copy()
            # tform = transform.EuclideanTransform(
            #     rotation = 0,
            #     translation = (df_particles2.loc[indx]['rlnOriginXAngst']/pixel_size, df_particles2.loc[indx]['rlnOriginYAngst']/pixel_size)
            #     )
            # tf_img = transform.warp(image, tform.inverse)
            image_seg = image > 0
            micrograph_segment[y-diameter:y+diameter,x-diameter:x+diameter] += image_seg

        if MICROGRAPH_DIR:
            img = Image.fromarray(np.uint8(255 * micrograph))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{MICROGRAPH_DIR}/{name}.png")

        if PROCESSED_MICROGRAPH_DIR:
            with mrcfile.open(f"{PROCESSED_DIR}/micrographs/{name}.mrc", permissive=True) as mrcs:
                processed_micrograph = mrcs.data
            img = Image.fromarray(np.uint8(255 * processed_micrograph))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{PROCESSED_MICROGRAPH_DIR}/{name}.png")

        if GROUND_TRUTH_DIR:
            img = Image.fromarray(np.uint8(255 * micrograph_segment))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{GROUND_TRUTH_DIR}/{name}.png")

        if plot:
            _, ax = plt.subplots(figsize=(12, 12))

            if plot_processed:
                ax.imshow(simple_micrograph_preprocessing(processed_micrograph), cmap='gray')
            else:
                ax.imshow(simple_micrograph_preprocessing(micrograph), cmap='gray')
            ax.imshow(micrograph_segment, cmap='inferno', alpha=0.4)

            for indx in locations.index:
                x, y = locations.loc[indx][['x_coord', 'y_coord']]
                c = matplotlib.patches.Circle((x, y), radius=radius, fill=False, color='r')
                ax.add_patch(c)

            plt.show()

    image_names = keep2
    for name in image_names:
        name = name[:-4]
        with mrcfile.open(f"{EMPIAR_DIR}/micrographs/{name}.mrc") as mrcs:
            micrograph = mrcs.data
        locations = labeled_val_particles[labeled_val_particles['image_name'] == name]
        micrograph_segment = np.zeros_like(micrograph, dtype=bool)

        for indx in locations.index:
            x, y = locations.loc[indx][['x_coord', 'y_coord']]
            image = particles2[indx].copy()
            # tform = transform.EuclideanTransform(
            #     rotation = 0,
            #     translation = (df_particles2_val.loc[indx]['rlnOriginXAngst']/pixel_size, df_particles2.loc[indx]['rlnOriginYAngst']/pixel_size)
            #     )
            # tf_img = transform.warp(image, tform.inverse)
            image_seg = image > 0
            micrograph_segment[y-diameter:y+diameter,x-diameter:x+diameter] += image_seg

        if MICROGRAPH_DIR:
            img = Image.fromarray(np.uint8(255 * micrograph))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{MICROGRAPH_DIR}/{name}.png")

        if PROCESSED_MICROGRAPH_DIR:
            with mrcfile.open(f"{PROCESSED_DIR}/micrographs/{name}.mrc", permissive=True) as mrcs:
                processed_micrograph = mrcs.data
            img = Image.fromarray(np.uint8(255 * processed_micrograph))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{PROCESSED_MICROGRAPH_DIR}/{name}.png")

        if GROUND_TRUTH_DIR:
            img = Image.fromarray(np.uint8(255 * micrograph_segment))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{GROUND_TRUTH_DIR}/{name}.png")

        if plot:
            _, ax = plt.subplots(figsize=(12, 12))

            if plot_processed:
                ax.imshow(simple_micrograph_preprocessing(processed_micrograph), cmap='gray')
            else:
                ax.imshow(simple_micrograph_preprocessing(micrograph), cmap='gray')
            ax.imshow(micrograph_segment, cmap='inferno', alpha=0.4)

            for indx in locations.index:
                x, y = locations.loc[indx][['x_coord', 'y_coord']]
                c = matplotlib.patches.Circle((x, y), radius=radius, fill=False, color='r')
                ax.add_patch(c)

            plt.show()
    image_names = keep3
    for name in image_names:
        name = name[:-4]
        if MICROGRAPH_DIR:
            img = Image.fromarray(np.uint8(255 * micrograph))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{MICROGRAPH_DIR}/{name}.png")

        if PROCESSED_MICROGRAPH_DIR:
            with mrcfile.open(f"{PROCESSED_DIR}/micrographs/{name}.mrc", permissive=True) as mrcs:
                processed_micrograph = mrcs.data
            img = Image.fromarray(np.uint8(255 * processed_micrograph))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{PROCESSED_MICROGRAPH_DIR}/{name}.png")
else:
    for name in image_names:
        with mrcfile.open(f"{EMPIAR_DIR}/micrographs/{name}.mrc") as mrcs:
            micrograph = mrcs.data
        locations = labeled_particles[labeled_particles['image_name'] == name]
        micrograph_segment = np.zeros_like(micrograph, dtype=bool)

        for indx in locations.index:
            x, y = locations.loc[indx][['x_coord', 'y_coord']]
            image = particles[indx].copy()
            tform = transform.EuclideanTransform(
                rotation = 0,
                translation = (df_particles2.loc[indx]['rlnOriginXAngst']/pixel_size, df_particles2.loc[indx]['rlnOriginYAngst']/pixel_size)
                )
            tf_img = transform.warp(image, tform.inverse)
            image_seg = segment(tf_img)
            micrograph_segment[y-diameter:y+diameter,x-diameter:x+diameter] += image_seg

        if MICROGRAPH_DIR:
            img = Image.fromarray(np.uint8(255 * micrograph))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{MICROGRAPH_DIR}/{name}.png")

        if PROCESSED_MICROGRAPH_DIR:
            with mrcfile.open(f"{PROCESSED_DIR}/micrographs/{name}.mrc", permissive=True) as mrcs:
                processed_micrograph = mrcs.data
            img = Image.fromarray(np.uint8(255 * processed_micrograph))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{PROCESSED_MICROGRAPH_DIR}/{name}.png")

        if GROUND_TRUTH_DIR:
            img = Image.fromarray(np.uint8(255 * micrograph_segment))  # no opencv required
            img.save(f"{RESULT_DIR}/dataset/{EMPIAR_ID}/{GROUND_TRUTH_DIR}/{name}.png")

        if plot:
            _, ax = plt.subplots(figsize=(12, 12))

            if plot_processed:
                ax.imshow(simple_micrograph_preprocessing(processed_micrograph), cmap='gray')
            else:
                ax.imshow(simple_micrograph_preprocessing(micrograph), cmap='gray')
            ax.imshow(micrograph_segment, cmap='inferno', alpha=0.4)

            for indx in locations.index:
                x, y = locations.loc[indx][['x_coord', 'y_coord']]
                c = matplotlib.patches.Circle((x, y), radius=radius, fill=False, color='r')
                ax.add_patch(c)

            plt.show()
"""

## Store EMPIAR data

In [ ]:
!rsync -av {EMPIAR_DIR}/micrographs/ {RESULT_DIR}/dataset/{EMPIAR_ID}/micrographs/
!rsync -av {EMPIAR_DIR}/ground_truth/ {RESULT_DIR}/dataset/{EMPIAR_ID}/ground_truth/
!rsync -av {EMPIAR_DIR}/particles_stack/ {RESULT_DIR}/dataset/{EMPIAR_ID}/particles_stack/

# Run CryoSparc

In [ ]:
# from pyngrok import ngrok, conf
# import getpass

In [ ]:
# print("Enter your authtoken, which can be copied from https://dashboard.ngrok.com/")
# conf.get_default().auth_token = getpass.getpass()

In [ ]:
# Setup a tunnel to the port 61000
# public_url = ngrok.connect(61000)

In [ ]:
 # public_url

In [ ]:
!rm -r /content/cryosparc_data

!mkdir /content/cryosparc_data
!cp /content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data/EMPIAR-10017/particles_stack/*.mrc /content/cryosparc_data/